## 0. Preparación (solo en Google Colab)

Esta celda no hace nada en local. En Colab clona el repositorio, entra en él e instala las dependencias con las versiones del proyecto (necesarias para que el modelo guardado sea compatible con la app). Si Colab pide reiniciar el entorno, reinícialo y vuelve a ejecutar desde aquí.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/Chefwise")
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/lezti02/Chefwise.git", str(REPO_DIR)])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 01 · Limpieza de recetas

**Entrada:** los CSV crudos de las dos fuentes (no están en el repo, están en Google Drive).
**Salida:** `data/processed/recetas_limpias.csv` (la receta que muestra la app).

| Fuente | Archivo crudo |
|---|---|
| RecetasDeLaAbuela ([Hugging Face](https://huggingface.co/datasets/somosnlp/RecetasDeLaAbuela)) | `recetasdelaabuela.csv` |
| Kiwilimon (scraping con `src/web_scrapping/`) | `recetas_kiwilimon.csv` |

Pasos: unir fuentes → arreglar tiempos y nombres → limpiar texto (letras rotas, marcas, promoción) → pasos de una sola instrucción → quitar duplicados → etiquetas de la app → validar y guardar.

## 1. Datos crudos desde Google Drive (sin iniciar sesión)

Los CSV crudos no están en el repo. Se descargan de Google Drive por **enlace público**, así que el notebook funciona desde cualquier cuenta de Colab (no se monta ningún Drive).

Los IDs de `DRIVE_FILE_IDS` son los de los archivos en Drive (compartidos como *Cualquier persona con el enlace → Lector*). Si cambias los archivos de lugar, pega el nuevo ID: es la parte central del enlace `https://drive.google.com/file/d/<ID>/view`.

Funciona igual en Colab y en local: los CSV se descargan a una carpeta temporal (`/content/raw` en Colab, la carpeta temporal del sistema en local) y **nunca** se guardan dentro del repo. El CSV limpio se guarda en `data/processed/` del repo (en Colab, el clon de la celda anterior).

In [ ]:
import ast
import csv
import html
import os
import re
import subprocess
import sys
import tempfile
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# ID de cada archivo en Drive (compartido como "Cualquier persona con el enlace").
DRIVE_FILE_IDS = {
    "recetasdelaabuela.csv": "1fAMN7GUbxOBeoJ130dujEoM6F9gOm0fv",
    "recetas_kiwilimon.csv": "106zPDRq1ySvYtjlglWsF7yGwOfpesCA6",
}

# Los CSV crudos se descargan siempre a una carpeta temporal (nunca dentro del repo).
RAW_DIR = Path("/content/raw") if "google.colab" in sys.modules else Path(tempfile.gettempdir()) / "chefWise_raw"
ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent       # raíz del repo
OUTPUT_PATH = ROOT / "data" / "processed" / "recetas_limpias.csv"


def download_raw():
    """Descarga de Drive los CSV que falten en RAW_DIR."""
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    for name, file_id in DRIVE_FILE_IDS.items():
        if (RAW_DIR / name).exists():
            continue
        assert not file_id.startswith("PEGA"), f"Falta el ID de Drive de {name} en DRIVE_FILE_IDS."
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            import gdown
        gdown.download(id=file_id, output=str(RAW_DIR / name), quiet=False)
        assert (RAW_DIR / name).exists(), f"No se pudo descargar {name}: revisa que el enlace sea público."


download_raw()
ABUELA_PATH = RAW_DIR / "recetasdelaabuela.csv"
KIWI_PATH = RAW_DIR / "recetas_kiwilimon.csv"

FUNNEL = []                                  # recetas que quedan tras cada paso


def log_rows(label, n):
    FUNNEL.append((label, int(n)))
    print(f"{label}: {int(n):,} recetas")

## 2. Lectura de las fuentes

In [ ]:
abuela_raw = pd.read_csv(ABUELA_PATH)
# En esta fuente "Tiempo" no son minutos sino el tipo de plato: se renombra.
abuela_raw = abuela_raw.rename(columns={"Tiempo": "meal_type"})
log_rows("RecetasDeLaAbuela leídas", len(abuela_raw))

In [ ]:
KIWI_COLUMNS = [
    "titulo", "dificultad", "tiempo", "tiempo_coccion",
    "ingredientes", "preparacion", "valoracion", "num_comentarios", "url",
]


def read_kiwilimon(path):
    # Devuelve (tabla con las filas buenas, lista de filas descartadas).
    # Las columnas salen de la cabecera del archivo; las que el scraper no trae quedan vacías.
    good_rows, bad_rows = [], []
    with path.open(encoding="utf-8-sig", newline="", errors="replace") as f:
        reader = csv.reader(f)
        first = next(reader)
        has_header = "titulo" in [c.strip().lower() for c in first]
        header = [c.strip().lower() for c in first] if has_header else list(KIWI_COLUMNS)
        rows = reader if has_header else [first, *reader]
        for line_number, row in enumerate(rows, start=2):
            if not row:
                continue                      # línea en blanco
            if [c.strip().lower() for c in row] == header:
                continue                      # cabecera repetida: se ignora
            if len(row) == len(header):
                good_rows.append(row)         # fila normal
            else:
                bad_rows.append((line_number, len(row), row[0][:60]))   # fila rota (dos recetas pegadas): se descarta

    table = pd.DataFrame(good_rows, columns=header)
    for col in KIWI_COLUMNS:
        if col not in table:
            table[col] = ""
    return table[KIWI_COLUMNS], bad_rows


# Kiwilimon: en el scraping una línea llegó a pegar dos recetas; esas filas rotas se descartan.
kiwi_raw, kiwi_bad_rows = read_kiwilimon(KIWI_PATH)
kiwi_raw = kiwi_raw.rename(columns={"tiempo": "prep_time_raw", "tiempo_coccion": "cook_time_raw"})
log_rows("Kiwilimon leídas", len(kiwi_raw))

## 3. Funciones de limpieza básicas (texto, tiempos, etiquetas sueltas)

In [ ]:
# Pares de letras rotas más comunes (Ã© -> é) y cómo se arreglan.
_MOJI_FIXED = (
    ("Ã¡", "á"), ("Ã©", "é"), ("Ãí", "í"), ("Ãó", "ó"), ("Ãú", "ú"),
    ("Ãñ", "ñ"), ("Ãü", "ü"), ("ÃÁ", "Á"), ("ÃÍ", "Í"),
    ("ÃÓ", "Ó"), ("ÃÚ", "Ú"), ("ÃÑ", "Ñ"), ("Ã ", "à"),
    ("Ã‰", "É"),  # Ã‰  (cp1252)
    ("Ãš", "Ú"),  # Ãš
)



# Arregla las letras rotas par por par, cuando el texto está mezclado (parte bien, parte rota).
def _repair_mojibake_pairs(text: str) -> str:
    """Corrige pares tipo Ã© mezclados con texto ya bien decodificado."""
    for src, dst in _MOJI_FIXED:
        if src in text:
            text = text.replace(src, dst)
    def repl(match):
        try:
            return match.group(0).encode("latin1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return match.group(0)
    text = re.sub("Ã[\xa0-\xff]", repl, text)
    text = re.sub("Â[\xa0-\xff ]", repl, text)
    return text


# Intenta arreglar todo el texto de una vez; si no se puede, lo hace par por par.
def _repair_mojibake_chunk(chunk: str) -> str:
    """Repara UTF-8 leído como Latin-1. Si falla el bloque completo, va por pares."""
    try:
        return chunk.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return _repair_mojibake_pairs(chunk)


# FUNCIÓN BASE: devuelve el texto limpio (sin letras rotas ni espacios dobles). Si el valor está vacío devuelve "".
def fix_text_encoding(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).replace("\ufeff", "").strip()
    if any(marker in text for marker in ("Ã", "Â", "â", "ðŸ", "�")):
        text = _repair_mojibake_chunk(text)
        if "Ã" in text or "Â" in text:
            text = _repair_mojibake_pairs(text)
    return re.sub(r"\s+", " ", text).strip()


# Si el texto parece una lista de Python ("['a', 'b']") la convierte en una lista real.
def parse_list_like(value):
    text = fix_text_encoding(value)
    if not text or text == "[]":
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [fix_text_encoding(x) for x in parsed if fix_text_encoding(x)]
        except (ValueError, SyntaxError):
            pass
    return None


# Une varias partes con " | " y quita las que estén vacías.
def join_parts(parts) -> str:
    return " | ".join(p for p in (fix_text_encoding(x) for x in parts) if p)


# Ingredientes: deja siempre el formato "ingrediente 1 | ingrediente 2 | ...".
def clean_ingredient_text(value) -> str:
    text = fix_text_encoding(value)
    parsed = parse_list_like(text)
    if parsed is not None:
        return join_parts(parsed)
    if "|" in text:
        return join_parts(text.split("|"))
    return text


# Pasos: quita "Paso 1.", "1)" y deja "paso | paso | paso".
def clean_instruction_text(value) -> str:
    text = fix_text_encoding(value)
    parsed = parse_list_like(text)
    if parsed is not None:
        cleaned = [re.sub(r"^\d+[\).:\-]\s*", "", p) for p in parsed]
        return join_parts(cleaned)
    if "|" in text:
        return join_parts(text.split("|"))
    if re.search(r"\bPaso\s*\d+", text, flags=re.I):
        parts = re.split(r"\bPaso\s*\d+\s*[.:\-]?\s*", text, flags=re.I)
        return join_parts(p.strip(" ,;") for p in parts)
    return text


# Texto "comparable": minúsculas, sin acentos ni símbolos. Sirve para comparar y buscar duplicados.
def normalize_key(value) -> str:
    text = fix_text_encoding(value).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
# Convierte "1 h 30 min", "45 mins", "2 hrs"... a minutos (número).
def parse_minutes(value):
    text = fix_text_encoding(value).lower()
    if not text:
        return np.nan
    text = text.replace("hrs", "h").replace("hr", "h")
    text = text.replace("mins", "m").replace("minutos", "m").replace("minuto", "m").replace("min", "m")
    text = text.replace(" ", "")
    hours = re.search(r"(\d+(?:\.\d+)?)h", text)
    minutes = re.search(r"(\d+(?:\.\d+)?)m", text)
    total = 0.0
    found = False
    if hours:
        total += float(hours.group(1)) * 60
        found = True
    if minutes:
        total += float(minutes.group(1))
        found = True
    if found:
        return total
    only = re.fullmatch(r"(\d+(?:\.\d+)?)", text)
    return float(only.group(1)) if only else np.nan


def parse_rating(value):
    """RecetasDeLaAbuela guarda 'Valoración: 5 (1 voto)'. Nos quedamos solo con el 5."""
    if pd.isna(value):
        return np.nan
    text = fix_text_encoding(value).replace(",", ".")
    match = re.search(r"Valoraci[oó]n\s*:\s*(\d+(?:\.\d+)?)", text, flags=re.I)
    return float(match.group(1)) if match else np.nan

In [ ]:
# Convierte las categorías (lista o texto con comas) en "cat 1 | cat 2".
def parse_tags(value) -> str:
    text = fix_text_encoding(value)
    parsed = parse_list_like(text)
    if parsed is not None:
        return join_parts(parsed)
    if not text or text == "[]":
        return ""
    return join_parts(text.split(","))


# Unifica cómo se escribe el país ("mexico" -> "México", "peru" -> "Perú"...).
def normalize_country(value) -> str:
    text = fix_text_encoding(value)
    key = normalize_key(text)
    mapping = {
        "mexico": "México",
        "peru": "Perú",
        "puerto rico": "Puerto Rico",
        "republica dominicana": "República Dominicana",
        "el salvador": "El Salvador",
        "costa rica": "Costa Rica",
        "espana": "España",
        "chile": "Chile",
        "colombia": "Colombia",
        "argentina": "Argentina",
        "venezuela": "Venezuela",
        "ecuador": "Ecuador",
        "guatemala": "Guatemala",
        "uruguay": "Uruguay",
        "bolivia": "Bolivia",
        "nicaragua": "Nicaragua",
        "cuba": "Cuba",
        "internacional": "Internacional",
    }
    return mapping.get(key, text) if text else ""


# Saca el nombre del sitio a partir de la URL.
def source_from_url(url: str) -> str:
    match = re.search(r"https?://([^/]+)", fix_text_encoding(url).lower())
    domain = match.group(1).replace("www.", "") if match else ""
    return {
        "recetasgratis.net": "RecetasGratis",
        "yanuq.com": "Yanuq",
        "cookpad.com": "Cookpad",
        "directoalpaladar.com": "Directo al Paladar",
        "elmueble.com": "El Mueble",
        "vitonica.com": "Vitónica",
        "kiwilimon.com": "Kiwilimon",
    }.get(domain, domain or "Desconocido")


# Kiwilimon mete la categoría en la URL (/receta/postres/...). Aquí la sacamos.
def root_kiwilimon_category(url: str) -> str:
    match = re.search(r"/receta/([^/]+)", fix_text_encoding(url).lower())
    if not match:
        return ""
    return match.group(1).replace("-", " ").strip()

# Todas las formas en que las fuentes escriben la dificultad -> 3 niveles de la app.
DIFFICULTY_MAP = {
    "muy baja": "facil", "baja": "facil", "facil": "facil",
    "media": "intermedio", "intermedio": "intermedio", "intermedia": "intermedio",
    "alta": "reto", "muy alta": "reto", "dificil": "reto", "reto": "reto",
}


def map_difficulty(value):
    """'Dificultad muy baja' -> 'facil'. Si no hay dato, devuelve vacío (NaN)."""
    text = re.sub(r"^dificultad\s*", "", fix_text_encoding(value), flags=re.I)
    return DIFFICULTY_MAP.get(normalize_key(text), np.nan)

## 4. Unir las fuentes con las mismas columnas

In [ ]:
abuela = pd.DataFrame()
abuela["name"] = abuela_raw["Nombre"].map(fix_text_encoding)
abuela["url"] = abuela_raw["URL"].map(fix_text_encoding)
abuela["ingredients"] = abuela_raw["Ingredientes"].map(clean_ingredient_text)
abuela["instructions"] = abuela_raw["Pasos"].map(clean_instruction_text)
abuela["source"] = abuela["url"].map(source_from_url)
abuela["country"] = abuela_raw["Pais"].map(normalize_country)
abuela["meal_type"] = abuela_raw["meal_type"].map(fix_text_encoding)        # antes 'Tiempo'
abuela["category_tags"] = abuela_raw["Categoria"].map(parse_tags)
abuela["difficulty"] = abuela_raw["Dificultad"].map(map_difficulty)
abuela["total_time_min"] = abuela_raw["Duracion"].map(parse_minutes)
abuela["rating"] = abuela_raw["Valoracion y Votos"].map(parse_rating)
abuela["prep_time_min"] = np.nan     # esta fuente solo da el total
abuela["cook_time_min"] = np.nan

In [ ]:
kiwi = pd.DataFrame()
kiwi["name"] = kiwi_raw["titulo"].map(fix_text_encoding)
kiwi["url"] = kiwi_raw["url"].map(fix_text_encoding)
kiwi["ingredients"] = kiwi_raw["ingredientes"].map(clean_ingredient_text)
kiwi["instructions"] = kiwi_raw["preparacion"].map(clean_instruction_text)
kiwi["source"] = "Kiwilimon"
kiwi["country"] = "México"
kiwi["meal_type"] = kiwi["url"].map(root_kiwilimon_category)       # ej. "postres", "bebidas"
kiwi["category_tags"] = ""
kiwi["difficulty"] = kiwi_raw["dificultad"].map(map_difficulty)
kiwi["prep_time_min"] = kiwi_raw["prep_time_raw"].map(parse_minutes)
kiwi["cook_time_min"] = kiwi_raw["cook_time_raw"].map(parse_minutes)
kiwi["total_time_min"] = kiwi[["prep_time_min", "cook_time_min"]].sum(axis=1, min_count=1)
kiwi["rating"] = pd.to_numeric(kiwi_raw["valoracion"], errors="coerce")

In [ ]:
COMMON_COLUMNS = [
    "source", "name", "url", "ingredients", "instructions", "country",
    "meal_type", "category_tags", "difficulty",
    "prep_time_min", "cook_time_min", "total_time_min", "rating",
]

recipes = pd.concat([abuela[COMMON_COLUMNS], kiwi[COMMON_COLUMNS]], ignore_index=True)

text_columns = ["source", "name", "url", "ingredients", "instructions", "country",
                "meal_type", "category_tags"]
for col in text_columns:
    recipes[col] = recipes[col].map(fix_text_encoding)

optional_text = ["country", "meal_type", "category_tags", "instructions"]
for col in optional_text:
    recipes[col] = recipes[col].replace("", np.nan)

log_rows("Recetas unidas", len(recipes))

missing_name = recipes["name"].eq("")
missing_ingredients = recipes["ingredients"].eq("")
recipes = recipes.loc[~missing_name & ~missing_ingredients].copy()
log_rows("Con nombre e ingredientes", len(recipes))

## 5. Nombres y tiempos

In [ ]:
# Palabras que van en minúscula dentro de un título ("Arroz con Pollo").
_SMALL = {"de", "del", "la", "el", "los", "las", "y", "o", "u", "con", "en", "al",
          "a", "un", "una", "para", "por", "sin", "the", "and"}


# Pasa el nombre a formato Título y quita el "Cómo hacer..." del inicio.
def title_es(name: str) -> str:
    text = fix_text_encoding(name)
    text = re.sub(r"^¿?c[oó]mo hacer\s+", "", text, flags=re.I).strip(" ¿?")
    letters = [c for c in text if c.isalpha()]
    # Nombres casi todo en minúsculas o casi todo en MAYÚSCULAS -> formato Título.
    mostly_lower = letters and sum(c.islower() for c in letters) / len(letters) > 0.8
    mostly_upper = letters and sum(c.isupper() for c in letters) / len(letters) > 0.8
    if mostly_lower or mostly_upper:
        words = re.split(r"(\s+|-)", text)
        out = []
        first_word = True
        for w in words:
            if not w or re.fullmatch(r"\s+|-", w):
                out.append(w)
                continue
            low = w.lower()
            if (not first_word) and low in _SMALL:
                out.append(low)
            else:
                out.append(low[:1].upper() + low[1:] if low else w)
            first_word = False
        text = "".join(out)
    return text


def fix_total_time(row):
    """Devuelve el tiempo total usable en minutos (o NaN si no es confiable)."""
    prep, cook, total = row["prep_time_min"], row["cook_time_min"], row["total_time_min"]

    if pd.notna(prep) and prep == 0:
        prep = np.nan
    if pd.notna(cook) and cook == 0:
        cook = np.nan

    if row["source"] == "Kiwilimon":
        # "30h" casi siempre eran 30 minutos mal escritos.
        if pd.notna(prep) and prep >= 25 * 60:
            hours = prep / 60.0
            prep = np.nan if hours > 96 else hours
        parts = [x for x in (prep, cook) if pd.notna(x)]
        total = float(sum(parts)) if parts else np.nan

    if pd.notna(total) and total == 0:
        total = np.nan
    if pd.notna(total) and total > 12 * 60:
        total = np.nan
    return total

In [ ]:
# Texto residual: letras rotas que sobrevivieron.
for col in ["name", "ingredients", "instructions"]:
    recipes[col] = recipes[col].map(lambda x: x if pd.isna(x) else fix_text_encoding(x))

recipes["name"] = recipes["name"].map(title_es)

recipes["total_time_min"] = recipes.apply(fix_total_time, axis=1)
recipes = recipes.drop(columns=["prep_time_min", "cook_time_min"])     # ya no hacen falta

## 6. Limpieza profunda de texto
Listas de Python → texto, letras rotas, emojis, HTML, marcas comerciales → genérico, menciones a Kiwilimón y autopromoción.

In [ ]:
# Contadores para el reporte final: cuántas veces se aplicó cada regla.
STATS = Counter()
AUDIT = Counter()       # marcas con ®/™ que NO estaban en nuestras listas
AUDIT_EX = {}
CHANGES = {}            # cuántas celdas cambió cada paso: {(paso, columna): n}


# Acomoda espacios y puntuación después de borrar o cambiar palabras (para que no queden "( )" ni ", ,").
def tidy(t: str) -> str:
    """Limpieza de espacios/puntuación; solo se aplica a textos ya modificados."""
    segs = []
    for s in t.split(" | "):
        s = re.sub(r"[ \t\u00a0]{2,}", " ", s)
        s = re.sub(r"\(\s*[,;.:\-]?\s*\)", "", s)
        s = re.sub(r"\s+([,.;:!?])", r"\1", s)
        s = re.sub(r"([,;:])(\s*\1)+", r"\1", s)
        s = re.sub(r",\s*([.;:!?])", r"\1", s)
        s = re.sub(r"\b(?:de|del|con|para|y)\s*(?=[,.;:!?)]|$)", "", s, flags=re.I)
        s = re.sub(r"[ \t]{2,}", " ", s)
        s = re.sub(r",\s*$", "", s)
        s = s.strip(" ,;")
        if s:
            segs.append(s)
    return " | ".join(segs)


# ¿La posición está al inicio de una oración? Sirve para poner mayúscula después de un reemplazo.
def _is_start(s: str, pos: int) -> bool:
    pre = s[:pos].rstrip()
    return (not pre) or pre[-1] in "|.!?:¡¿("


# Hace un reemplazo con regex, lo cuenta (STATS) y cuida las mayúsculas.
def rsub(rx, repl, t, step, label):
    """re.sub con conteo de ocurrencias y capitalización según contexto."""
    def f(m):
        r = repl(m) if callable(repl) else m.expand(repl)
        if r and _is_start(m.string, m.start()):
            r = r[0].upper() + r[1:]
        return r
    t2, n = rx.subn(f, t)
    if n:
        STATS[(step, label)] += n
    return t2


# Si se borra un paso numerado, vuelve a numerar 1, 2, 3... sin huecos.
def renumber(t: str) -> str:
    segs = [s for s in t.split(" | ") if s.strip()]
    if segs and all(re.match(r"^\d+\s", s) for s in segs):
        segs = [f"{i} {re.sub(r'^\d+\s+', '', s)}" for i, s in enumerate(segs, 1)]
    return " | ".join(segs)


def apply_step(df, step, cols, func):
    """Aplica func(texto) a cada columna y anota cuántas celdas cambiaron."""
    for col in cols:
        old = df[col]
        new = old.map(lambda t: func(t) if isinstance(t, str) else t)
        changed = int((old.notna() & new.ne(old)).sum())
        CHANGES[(step, col)] = CHANGES.get((step, col), 0) + changed
        df[col] = new

In [ ]:
# Convierte "['a', 'b']" en una lista de Python (también si la lista está cortada o mal cerrada).
def parse_pylist(t: str):
    s = t.strip()
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return [str(x) for x in v]
    except Exception:
        pass
    inner = s.lstrip("[").rstrip("]").strip()           # lista truncada/malformada
    inner = inner.strip("'\" ")
    return re.split(r"""['"]\s*,\s*['"]""", inner)


# PASO 1 (ingredientes): si todavía quedó una lista como texto, la vuelve "a | b".
def desparse_ingredients(t):
    if not re.match(r"^\s*\[", t):
        return t
    items = [re.sub(r"\s+", " ", i).strip() for i in parse_pylist(t)]
    STATS[("5", "ingredients: lista -> ' | '")] += 1
    return " | ".join(i for i in items if i)


# PASO 1 (pasos): igual que arriba, y además quita el número del inicio.
def desparse_instructions(t):
    if not re.match(r"^\s*\[", t):
        return t
    items = [re.sub(r"\s+", " ", i).strip() for i in parse_pylist(t)]
    items = [re.sub(r"^\d+[.)]?\s+", "", i) for i in items if i]
    STATS[("5", "instructions: lista -> numerada")] += 1
    return " | ".join(f"{n} {i}" for n, i in enumerate(items, 1))


# ----------------------------------------------------------------------------
# PASO 2: letras rotas (mojibake), emojis, HTML, '<' '>'
# ----------------------------------------------------------------------------
# Letras que aparecen cuando el texto UTF-8 se leyó mal (Ã, Â, â...). Con ellas armamos la regex que las detecta.
_CONT = "".join(
    chr(b) if b in (0x81, 0x8D, 0x8F, 0x90, 0x9D) else bytes([b]).decode("cp1252")
    for b in range(0x80, 0xC0)
)
# Detecta una letra rota (por ejemplo "Ã©" que debería ser "é").
MOJI_RE = re.compile(
    rf"[ÂÃ][{_CONT}]|â[{_CONT}]{{2}}|ï[{_CONT}]{{2}}|ð[{_CONT}]{{3}}|[ÂÃâïð][{_CONT}]{{0,3}}"
)
# Detecta emojis y pictogramas.
EMOJI_RE = re.compile(
    "[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\u2300-\u23FF"
    "\uFE0E\uFE0F\u200D\u20E3\U000E0020-\U000E007F]+"
)
# Caracteres invisibles que ensucian el texto.
INVISIBLE_RE = re.compile("[\u200b-\u200f\u202a-\u202e\u2060\ufeff]")
# Entidades HTML conocidas (&nbsp;, &hellip;...) y por qué carácter se cambian.
ENT_MAP = {"&#8206;": "", "&#8207;": "", "&#8243;": '"', "&#8242;": "'",
           "&#133;": "...", "&hellip;": "...", "&nbsp;": " ", "&#160;": " "}
ENT_RE = re.compile(r"&(?:#\d+|#x[0-9a-fA-F]+|[A-Za-z][A-Za-z0-9]*);")
# Etiquetas HTML como <br> o <b>.
TAG_RE = re.compile(r"</?(?:br|p|b|i|u|em|strong|span|div|li|ul|ol|a|img)\b[^>]*>", re.I)
# Comillas y guiones "de imprenta" que copió el scraping. Los cambiamos por los normales
# para que el mismo ingrediente no se escriba de dos formas distintas (“ajo” y "ajo").
TYPOGRAPHIC = {
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u00ab": '"', "\u00bb": '"', "\u2033": '"',
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u2032": "'", "\u00b4": "'",
    "\u2013": "-", "\u2014": "-", "\u2212": "-", "\u2026": "...",
}
# Fracciones "bonitas" (½, ¼, ⅓...) a números de toda la vida. Importa porque el notebook 02 lee la
# cantidad del principio del renglón: "1½ tazas" se vuelve "1 1/2 tazas" y sí la reconoce.
FRACTIONS = {
    "\u00bd": " 1/2 ", "\u2153": " 1/3 ", "\u2154": " 2/3 ", "\u00bc": " 1/4 ", "\u00be": " 3/4 ",
    "\u2155": " 1/5 ", "\u2156": " 2/5 ", "\u2157": " 3/5 ", "\u2158": " 4/5 ",
    "\u2159": " 1/6 ", "\u215a": " 5/6 ", "\u2150": " 1/7 ", "\u215b": " 1/8 ",
    "\u215c": " 3/8 ", "\u215d": " 5/8 ", "\u215e": " 7/8 ", "\u2151": " 1/9 ", "\u2152": " 1/10 ",
    "\u2103": " °C", "\u2109": " °F",
}
# Adornos que copió el scraping (viñetas, flechas, símbolos sueltos). No dicen nada: fuera.
DECORATIVE_RE = re.compile(
    "[\u2022\u25aa\u25ab\u25cf\u25cb\u2023\u2043\u00b7\u2219\u00a4\u00ac\u00a8\u02da"
    "\u2192\u2190\u2191\u2193\u21d2\u21d0\u203a\u2039\u2713\u2714\u2605\u2606"
    "\u25b6\u25c0\u2756\u273f\u00a6\x81\x8d\x8f\x90\x9d]+")
# El guion "suave" es invisible y parte las palabras: se borra sin dejar espacio.
SOFT_HYPHEN = "\u00ad"


# Repara una letra rota. Si no se puede recuperar, la elimina.
def _moji_fix(m):
    run = m.group(0)
    bs = bytearray()
    for ch in run:
        try:
            bs += ch.encode("cp1252")
        except UnicodeEncodeError:
            if ord(ch) < 256:
                bs.append(ord(ch))
            else:
                return ""
    try:
        return bytes(bs).decode("utf-8").replace("\xa0", " ")
    except UnicodeDecodeError:
        return ""         # secuencia irrecuperable (bytes perdidos) -> se elimina


# Cambia letras "matemáticas" raras (𝐀𝐛𝐜) por letras normales.
def _math_to_plain(s):
    return "".join(unicodedata.normalize("NFKC", c) if 0x1D400 <= ord(c) <= 0x1D7FF else c for c in s)


# PASO 2: limpieza profunda de encoding, emojis, HTML y símbolos invisibles.
def fix_encoding(t):
    o = t
    t2, n = MOJI_RE.subn(_moji_fix, t)
    if n:
        STATS[("4", "mojibake reparado/eliminado")] += n
    t = _math_to_plain(t2)
    t2, n = EMOJI_RE.subn("", t)
    if n:
        STATS[("4", "emojis/pictogramas eliminados")] += n
    t = t2
    for k, v in ENT_MAP.items():
        if k in t:
            STATS[("4", "entidad HTML " + k)] += t.count(k)
            t = t.replace(k, v)
    if ENT_RE.search(t):
        t, n = ENT_RE.subn(lambda m: html.unescape(m.group(0)), t)
        STATS[("4", "otras entidades HTML")] += n
    t, n = INVISIBLE_RE.subn("", t)
    if n:
        STATS[("4", "caracteres invisibles")] += n
    t, n = TAG_RE.subn(" ", t)
    if n:
        STATS[("4", "etiquetas HTML")] += n
    for raro, normal in TYPOGRAPHIC.items():
        if raro in t:
            STATS[("4", "comilla/guion tipográfico")] += t.count(raro)
            t = t.replace(raro, normal)
    for raro, normal in FRACTIONS.items():
        if raro in t:
            STATS[("4", "fracción/símbolo a texto normal")] += t.count(raro)
            t = t.replace(raro, normal)
    if SOFT_HYPHEN in t:
        STATS[("4", "guion suave invisible")] += t.count(SOFT_HYPHEN)
        t = t.replace(SOFT_HYPHEN, "")
    t2, n = DECORATIVE_RE.subn(" ", t)
    if n:
        STATS[("4", "viñetas y flechas de adorno")] += n
    t = t2
    # Un acento suelto pegado a una letra (e + ´) se junta en una sola letra (é).
    t = unicodedata.normalize("NFC", t)
    return tidy(t) if t != o else t


# PASO 2: a veces el '<' o '>' se usó como separador en vez de coma; aquí se arregla.
def fix_angle(t):
    if "<" not in t and ">" not in t:
        return t
    t = t.replace("<3", "\x00H\x00")           # corazón ASCII: se conserva
    t = re.sub(r"\bve<", "vez", t)             # errata detectada: 've<' = 'vez'

    def f(m):
        s, pre = m.string, (m.string[m.start() - 1] if m.start() else "")
        if m.start() == 0:
            return ""
        if m.end() == len(s):
            return "" if pre in ".!?" else "."
        return " " if pre in ".,;:!?" else ", "

    out = []
    for seg in t.split(" | "):
        seg2, n = re.subn(r"\s*[<>]+\s*", f, seg)
        STATS[("4", "'<' '>' usados como separador")] += n
        out.append(seg2)
    return tidy(" | ".join(out)).replace("\x00H\x00", "<3")

In [ ]:
# Símbolo ® o ™ opcional después de la marca.
M = r"(?:\s?[®™])?"       # símbolo opcional tras la marca
# Palabras que suelen ir antes de una marca ("de la marca", "tipo").
PRE = r"(?:\b(?:de\s+la\s+marca|de\s+marca|marca|de|del|tipo)\s+)?"


# Une una lista de nombres en una sola regex ("a|b|c").
def _alt(names):
    return "|".join(sorted(names, key=len, reverse=True))


# Marcas ambiguas (palabras comunes): solo se tocan si van seguidas de ® / ™
# Marcas AMBIGUAS: solo se quitan si llevan ® o ™ (por ejemplo "Gloria®").
WITH_MARK = [
    r"D['’]?Gari", r"Gari", r"San\s+Juan", r"Tres\s+Estrellas", r"Del\s+Fuerte", r"Santa\s+Clara",
    r"Primavera", r"Extra\s+Special", r"Extra\s+Especial", r"Huichol", r"D[eo]l?\s+Campo", r"Domino",
    r"Berries\s+Paradise", r"San\s+Marcos", r"Gloria", r"Tortillinas", r"Luigi", r"Chantilly", r"Eva",
    r"Langa", r"Dolores", r"Ades", r"Evenflo(?:\s+Advance)?", r"Virgen\s+In[eé]s", r"Clavel", r"Iberia",
    r"Singles", r"Capullo", r"Ol[eé]ico", r"Plenia", r"Emergen-C", r"Silk", r"Zumba\s+Pica", r"Sos",
    r"Kinder", r"La\s+Moderna", r"Mark", r"Lybra", r"Bormioli", r"Saníssimo", r"Not\s+Butter",
    r"Cocina\s+Mestiza", r"Hummus", r"Caperucita", r"Valle\s+Frut", r"Fuerte\s+Origen",
]
# Marcas con ®/™ que se reemplazan por un genérico.
WITH_MARK_REPL = [          # (patrón, reemplazo genérico) — requieren ® / ™
    (r"PAM|Pam", "aceite en aerosol"),
    (r"Sidral\s+Mundet", "refresco de manzana"),
    (r"Reddi\s+Wip", "crema batida en aerosol"),
    (r"(?:[Oo]lla\s+de\s+)?Presi[oó]n\s+Express", "olla de presión"),
]
# Marcas inequívocas: se eliminan siempre (lista del usuario + hallazgos del análisis)
# Marcas que se borran SIEMPRE (alimentos).
ALWAYS_USER = [r"McCormick", r"Great\s+Value", r"Bimbo", r"Nestl[eéè]", r"Barilla", r"Heinz",
               r"Quaker", r"Danone", r"Alpina", r"Herdez"]
# Más marcas de alimentos que se borran siempre.
ALWAYS_FOOD = [r"D['’]Gari", r"Gamesa", r"Nutrioli", r"Oikos", r"Fage", r"Kikkoman", r"Yoplait",
               r"Danette", r"Skwinkles", r"Libanius", r"Jumex", r"Clemente\s+Jacques", r"La\s+Coste[ñn]a",
               r"Embasa", r"Valle\s+Frut", r"La\s+Asturiana", r"T[ií]a\s+Rosa", r"Do[ñn]a\s+Mar[ií]a",
               r"La\s+Botanera", r"La\s+Moderna", r"Best\s+Foods", r"Hellmann['’]?s", r"Kraft", r"Lala",
               r"Calahua", r"Proan", r"Del\s+Monte", r"Rexal"]
# Marcas de electrodomésticos y utensilios que se borran siempre.
ALWAYS_APPLIANCE = [r"Oster", r"Braun", r"Vitamix", r"Weber", r"T-?fal", r"Tefal", r"KitchenAid",
                    r"Nutri\s?bullet", r"Instant\s+Pot", r"Ninja", r"Cuisinart", r"Magimix", r"Moulinex",
                    r"Cecotec", r"Crock-?\s?Pot", r"Royal\s+Prestige", r"Taurus", r"Philips",
                    r"Samsung(?:\s+Family\s*Hub)?"]

RX_WITH_MARK = re.compile(rf"{PRE}\b(?:{_alt(WITH_MARK)})\s?[®™]")
RX_WITH_MARK_REPL = [(re.compile(rf"\b(?:{p})\s?[®™]"), r) for p, r in WITH_MARK_REPL]
RX_ALWAYS_USER = re.compile(rf"{PRE}\b(?:{_alt(ALWAYS_USER)})\b{M}", re.I)
RX_ALWAYS_FOOD = re.compile(rf"{PRE}\b(?:{_alt(ALWAYS_FOOD)})\b{M}", re.I)
RX_ALWAYS_APP = re.compile(rf"{PRE}\b(?:{_alt(ALWAYS_APPLIANCE)})\b{M}", re.I)

# True: si una palabra con mayúscula va pegada a ®/™ y no está en las listas, también se quita.
AUTO_STRIP = True      # elimina el token capitalizado que precede a ®/™ si no está en las listas
RX_AUTO = re.compile(rf"{PRE}\b[A-ZÁÉÍÓÚÑ][^\s®™]*\s?[®™]")
RX_RESIDUAL = re.compile(r"((?:[A-ZÁÉÍÓÚÑ][^\s®™]*\s+){0,2}[A-ZÁÉÍÓÚÑ][^\s®™]*)\s?[®™]")

# Thermomix y sus modelos (TM5, TM31...).
THERMO = r"Thermomix(?:\s*(?:TM\s?-?\s?\d{1,2}\b|TM\b|T\d{2}\b))?"


# Knorr/Maggi/Avecrem -> "caldo en polvo" (cuidando que no se repita la palabra).
# Palabras que indican que lo que sigue (o lo que viene antes) YA es el producto.
_CALDO_PRODUCTO = r"caldo|consom|sazonador|sopa|salsa|pur[eé]|concentrado|crema|mayonesa|aderezo|jugo"


def _knorr(m):
    had_de = bool(re.match(r"(?:de|del)\s", m.group(0), re.I))
    tomate = bool(re.search(r"tomate", m.group(0), re.I))
    pre = re.split(r"[|.;]", m.string[max(0, m.start() - 45):m.start()])[-1]
    post = m.string[m.end():m.end() + 40]
    # Caso "Knorr Caldo de Pollo": el producto viene DESPUÉS de la marca, así que
    # solo hay que borrar la marca. Si no, quedaría "caldo en polvo caldo de pollo".
    if re.match(rf"\s*(?:de\s+)?(?:{_CALDO_PRODUCTO})", post, re.I):
        return "de" if had_de else ""
    if re.search(r"\b(?:cubos?|cubitos?|pastillas?|sobres?|cuadritos?|cucharad(?:a|ita)s?|latas?|"
                 r"paquetes?|envases?|bolsas?)\s*$", pre, re.I):
        return "de caldo en polvo" + (" de tomate" if tomate else "")
    if re.search(r"caldo|consom|sazon|sopa|polvo|salsa|aderezo|crema|mayonesa|concentrado|pur[eé]|"
                 r"jugo|sabor", pre, re.I):
        return " de tomate" if tomate else ""
    return ("de " if had_de else "") + "caldo en polvo" + (" de tomate" if tomate else "")


# Royal -> "polvo para hornear".
def _royal(m):
    return m.group(1) if m.group(1) else "polvo para hornear"


# Lista de reemplazos: (descripción, regex, texto genérico).
REPLACEMENTS = [
    # (etiqueta, regex, reemplazo)
    ("Philadelphia -> queso crema",
     re.compile(rf"(?:queso|crema)(?:\s+(?:crema|de\s+queso))?\s+(?:tipo\s+)?(?:de\s+)?Philadelphia{M}", re.I), "queso crema"),
    ("Philadelphia -> queso crema",
     re.compile(rf"(?<!estilo )(?<!estilo de )(?<!ciudad de )(?:tipo\s+)?Philadelphia{M}", re.I), "queso crema"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\bde\s+(?:la\s+)?{THERMO}{M}", re.I), "del robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\bla\s+{THERMO}{M}", re.I), "el robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\buna\s+{THERMO}{M}", re.I), "un robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"\b(en|con|para|desde|sin|al|usando)\s+{THERMO}{M}", re.I), r"\1 el robot de cocina"),
    ("Thermomix -> robot de cocina",
     re.compile(rf"{THERMO}{M}", re.I), "robot de cocina"),
    ("Mycook/Bimby/Cookeo -> robot de cocina",
     re.compile(rf"\b(?:Mycook(?:\s+(?:Touch|Easy|Pro))?(?:\s+de\s+Taurus)?|Bimby|Cookeo|Thermochef){M}", re.I), "robot de cocina"),
    ("Maizena/Maicena -> fécula de maíz (dup.)",
     re.compile(rf"(f[eé]cula\s+de\s+ma[ií]z)\s+(?:marca\s+)?(?:Maizena|Maicena){M}", re.I), r"\1"),
    ("Maizena/Maicena -> fécula de maíz",
     re.compile(rf"\b(?:Maizena|Maicena){M}", re.I), "fécula de maíz"),
    ("Nutella -> crema de avellanas y cacao",
     re.compile(rf"(?:crema\s+de\s+avellanas?(?:\s+y\s+cacao)?\s+|chocolate\s+para\s+untar\s+)?(?:tipo\s+)?Nutella{M}", re.I),
     "crema de avellanas y cacao"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"\bgalleta\s+(?:tipo\s+)?Oreo{M}", re.I), "galleta de chocolate rellena"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"\b(?:1|una?)\s+Oreo{M}\b", re.I), "1 galleta de chocolate rellena"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"(?:galletas?|galletitas?)\s+(?:tipo\s+)?Oreos?{M}", re.I), "galletas de chocolate rellenas"),
    ("Oreo -> galletas de chocolate rellenas",
     re.compile(rf"\bOreos?{M}", re.I), "galletas de chocolate rellenas"),
    ("Knorr/Maggi/Avecrem -> caldo en polvo",
     re.compile(rf"(?:\b(?:de|del|marca|de\s+la\s+marca|de\s+marca)\s+)?\b(?:Knorr|Maggis?|Maggui|Avecrem)\b{M}"
                r"(?:\s+(?:Suiza|Tomate))?", re.I), _knorr),
    ("Royal (polvo para hornear)",
     re.compile(r"\bRoyal\b(?=\s+(?:polvo\s+(?:para|de)\s+hornear|para\s+hornear|levadura))\s*"), ""),
    ("Royal (polvo para hornear)",
     re.compile(r"((?:polvo\s+(?:para|de)\s+hornear|levadura\s+qu[ií]mica)\s+)?(?<!Kir )\bRoyal\b"
                r"(?!\s+(?i:Prestige|Icing|Canin|Oak|Jelly|Gala))" + M), _royal),
    ("La Lechera -> leche condensada",
     re.compile(rf"(leche\s+condensada\s+)?\bLa\s+Lechera{M}", re.I),
     lambda m: "leche condensada" if not m.group(1) else m.group(1)),
    ("Splenda -> edulcorante",
     re.compile(rf"(edulcorante|endulzante)\s+(?:bajo\s+en\s+calor[ií]as\s+)?(?:de\s+)?E?Splenda{M}", re.I), r"\1"),
    ("Splenda -> edulcorante",
     re.compile(rf"\bE?splenda{M}", re.I), "edulcorante"),
    ("Samsung Smart Oven -> horno",
     re.compile(r"Smart\s+Oven(?:\s+HotBlast)?(?:\s+de\s+Samsung)?", re.I), "horno"),
    ("Maggie (salsa/jugo) -> eliminada",
     re.compile(r"(?<=salsa )Maggie|(?<=jugo )Maggie|(?<=sazonador )Maggie|(?<=caldo )Maggie", re.I), ""),
    ("demaicena -> de fécula de maíz",
     re.compile(r"\bde(?:maizena|maicena)\b", re.I), "de fécula de maíz"),
    ("Pyrex -> refractario (dup.)",
     re.compile(rf"\b(refractario|recipiente|molde|fuente|charola|vasija)\s+(?:de\s+)?Pyrex{M}", re.I), r"\1"),
    ("Silpat -> tapete de silicón (dup.)",
     re.compile(rf"\b(?:tapete|alfombrilla|l[aá]mina|hoja)\s+(?:de\s+)?Silpat{M}", re.I), "tapete de silicón"),
    ("Lékué -> recipiente de silicón (dup.)",
     re.compile(rf"\b(?:recipiente|estuche|molde)\s+(?:de\s+)?L[eé]ku[eé]{M}", re.I), "recipiente de silicón"),
    ("Pyrex -> refractario",
     re.compile(rf"\bPyrex{M}", re.I), "refractario"),
    ("Silpat -> tapete de silicón",
     re.compile(rf"\bSilpat{M}", re.I), "tapete de silicón"),
    ("Lékué -> recipiente de silicón",
     re.compile(rf"\bL[eé]ku[eé]{M}", re.I), "recipiente de silicón"),
    ("Teflon -> antiadherente",
     re.compile(rf"\b(?:de\s+)?Tefl[oó]n{M}", re.I), "antiadherente"),
    ("Sidral Mundet -> refresco de manzana",
     re.compile(rf"\bSidral(?:\s+Mundet)?{M}", re.I), "refresco de manzana"),
]
# Reglas de borrado, en el orden en que se aplican.
ALWAYS_RULES = [("marca secundaria eliminada (lista del usuario)", RX_ALWAYS_USER),
                ("marca de alimento eliminada (hallazgo)", RX_ALWAYS_FOOD),
                ("marca de electrodoméstico/utensilio eliminada", RX_ALWAYS_APP)]

# En títulos las marcas de caldo se quitan (no se reemplazan por "caldo en polvo")
# En los TÍTULOS las marcas de caldo se quitan (no se cambian por "caldo en polvo").
RX_KNORR_NAME = re.compile(rf"{PRE}\b(?:Knorr|Maggis?|Maggui|Avecrem)\b{M}(?:\s+(?:Suiza|Tomate))?", re.I)

In [ ]:
# Atajos de regex: B_ = inicio de palabra, E_ = fin de palabra, Q_ = apóstrofo opcional.
B_ = r"(?<![\wáéíóúñÁÉÍÓÚÑ])"      # inicio de palabra (también con acentos)
E_ = r"(?-i:(?![a-záéíóúñ0-9_])(?!(?<=[A-ZÁÉÍÓÚÑ])[A-ZÁÉÍÓÚÑ]))"   # fin de palabra; admite "srirachaMayonesa" pero no "FANTASIA"
Q_ = r"['’´`]?"                     # apóstrofo opcional (Baileys, M&M's, Hershey's)

# Tablas para corregir el artículo cuando el genérico cambia de género (el Milo -> la malta).
_ART_F = {"el": "la", "del": "de la", "al": "a la", "un": "una", "unos": "unas", "los": "las"}
_ART_M = {"la": "el", "de la": "del", "a la": "al", "una": "un", "unas": "unos", "las": "los"}
_SMALL_T = {"de", "del", "la", "el", "los", "las", "y", "o", "con", "en", "al", "a", "para", "sin", "un", "una"}

# Aquí se van guardando las reglas de la lista ampliada.
EXTRA = []        # (etiqueta, regex, reemplazo, alcance)   alcance: "all" | "text" (no títulos)
_TRIG = []        # patrones que activan el bloque; evita correr ~150 regex sobre textos sin marcas
_GENERICS = set()


# Crea la regla que cambia también el artículo ("el Milo" -> "la malta achocolatada").
def _article_rule(label, brand, generic, g, scope):
    table = _ART_F if g == "f" else _ART_M
    arts = "|".join(sorted((re.escape(a).replace(r"\ ", r"\s+") for a in table), key=len, reverse=True))
    rx = re.compile(rf"\b({arts})\s+(?:{brand}){E_}{M}", re.I)
    return (label, rx, lambda m: table[" ".join(m.group(1).lower().split())] + " " + generic, scope)


# Registra una marca nueva: add_brand(descripción, marca, genérico, género, cat=palabras que la suelen preceder).
def add_brand(label, brand, generic, g="m", cat=None, scope="all", plain=True):
    """Marca -> término genérico.

    `cat`: palabra(s) de categoría que suelen preceder a la marca ("licor Cointreau");
    se reemplaza toda la frase por el genérico para no duplicar ("licor licor de naranja").
    `g`: género del genérico, para corregir el artículo ("el Milo" -> "la malta achocolatada").
    """
    _TRIG.append(brand)
    if generic and " " in generic:
        _GENERICS.add(generic)
    if cat:
        EXTRA.append((label, re.compile(
            rf"\b(?:{cat})\s+(?:(?:de|del|marca|tipo|estilo)\s+)?(?:{brand}){E_}{M}", re.I), generic, scope))
    if not plain:
        return
    if generic:
        EXTRA.append(_article_rule(label, brand, generic, g, scope))
        EXTRA.append((label, re.compile(rf"{B_}(?:{brand}){E_}{M}", re.I), generic, scope))
    else:
        EXTRA.append((label, re.compile(rf"{PRE}{B_}(?:{brand}){E_}{M}", re.I), "", scope))


# Registra una regla especial escrita a mano.
def add_special(label, pattern, repl, scope="all", flags=re.I, trig=None):
    _TRIG.append(trig or pattern)
    EXTRA.append((label, re.compile(pattern, flags), repl, scope))


# Marcas de refresco de cola.
COLA = rf"(?:Coca[\s-]?Cola|Pepsi(?:[\s-]?Cola)?)(?:\s+(?:Zero|Light|Diet|Life))?"
SODA_CAT = r"refrescos?(?:\s+de\s+cola)?|gaseosas?(?:\s+de\s+cola)?|bebidas?(?:\s+gaseosas?)?(?:\s+de\s+cola)?|sodas?(?:\s+de\s+cola)?|colas?"

# A partir de aquí: la lista de marcas, por tipo de producto.
# ---- Refrescos --------------------------------------------------------------------
add_special("Coca-Cola/Pepsi + 'a la/de la' -> refresco de cola",
            rf"\b(a|de|con|en)\s+la\s+{COLA}{E_}{M}",
            lambda m: {"a": "al", "de": "del"}.get(m.group(1).lower(), m.group(1).lower() + " el") + " refresco de cola",
            trig=COLA)
add_special("Coca-Colas (plural) -> refrescos de cola", rf"{B_}(?:Coca[\s-]?Cola|Pepsi)s{E_}", "refrescos de cola", trig=r"(?:Coca[\s-]?Cola|Pepsi)s")
add_brand("Coca-Cola/Pepsi -> refresco de cola", COLA, "refresco de cola", "m", cat=SODA_CAT)
add_special("'lata de Coca' -> refresco de cola",
            rf"\b((?:lata|botella|vaso|taza|litro|mililitros?|ml)s?\s+(?:de\s+)?)(?-i:Coca)(?!\s*-?\s*Cola){E_}",
            r"\1refresco de cola", trig=r"(?-i:Coca)\b")
add_brand("Sprite/7Up -> refresco de limón", r"Sprite|7[\s-]?Up|Seven[\s-]?Up", "refresco de limón", "m", cat=SODA_CAT)
add_brand("Fanta -> refresco de naranja", r"Fanta", "refresco de naranja", "m", cat=SODA_CAT)
add_brand("Squirt -> refresco de toronja", r"Squirt", "refresco de toronja", "m", cat=SODA_CAT)
add_brand("Red Bull/Monster -> bebida energética", r"Red\s?Bull|Monster(?:\s+Energy)?", "bebida energética", "f",
          cat=r"bebidas?\s+energ[eé]ticas?")
add_brand("Lipton -> té", r"Lipton", "té", "m", cat=r"t[eé](?:\s+negro|\s+helado)?")
add_brand("Nestea -> té helado en polvo", r"Nestea", "té helado en polvo", "m", cat=r"t[eé](?:\s+helado)?")

# ---- Licores ---------------------------------------------------------------------
add_special("'tipo Nescafé' -> eliminado", rf"\s*\((?:tipo|como)\s+Nescaf[eé]\)", "", trig=r"Nescaf")
add_brand("Nescafé -> café instantáneo", r"Nescaf[eé]", "café instantáneo", "m", cat=r"caf[eé](?:\s+(?:soluble|instant[aá]neo))?")
add_brand("Baileys -> licor de crema irlandesa", rf"Bailey{Q_}s(?:\s+Irish\s+Cream)?(?:\s+Original)?",
          "licor de crema irlandesa", "m", cat=r"licor(?:\s+de\s+crema(?:\s+irlandesa)?)?|crema(?:\s+irlandesa)?")
add_brand("Cointreau/Grand Marnier -> licor de naranja", r"Cointreau|Cointrau|Grand\s+Marnier",
          "licor de naranja", "m", cat=r"licor(?:\s+de\s+naranja)?")
add_brand("Kahlúa/Tía María -> licor de café", r"Kahl[uú]a|T[ií]a\s+Mar[ií]a", "licor de café", "m",
          cat=r"licor(?:\s+de\s+caf[eé])?")
add_brand("Frangelico -> licor de avellana", r"Frangelico", "licor de avellana", "m",
          cat=r"licor(?:\s+de\s+avellanas?)?")
add_brand("Midori -> licor de melón", r"Midori", "licor de melón", "m", cat=r"licor(?:\s+de\s+mel[oó]n)?")
add_brand("Malibú -> licor de coco", r"Malib[uú]", "licor de coco", "m", cat=r"licor(?:\s+de\s+coco)?|ron(?:\s+de\s+coco)?")
add_brand("Chambord -> licor de frambuesa", r"Chambord", "licor de frambuesa", "m", cat=r"licor(?:\s+de\s+frambuesas?)?")
add_brand("Pernod -> licor de anís", r"Pernod", "licor de anís", "m", cat=r"licor(?:\s+de\s+an[ií]s)?")
add_brand("Licor 43 -> licor de vainilla", r"Licor\s+43|Cuarenta\s+y\s+Tres", "licor de vainilla", "m")
add_brand("Campari -> aperitivo amargo", r"Campari", "aperitivo amargo", "m", cat=r"aperitivo(?:\s+amargo)?")
add_brand("Bacardí/Havana Club -> ron", r"Bacard[ií](?:\s+(?:Blanco|Carta\s+Blanca|Limón))?|Havana\s+Club", "ron", "m", cat=r"ron")
add_brand("Absolut -> vodka", r"Absolut", "vodka", "m", cat=r"vodka")
add_brand("Tanqueray/Bombay -> ginebra", r"Tanqueray|Bombay(?:\s+Sapphire)?", "ginebra", "f", cat=r"ginebra")
add_brand("Jack Daniel's/Johnnie Walker -> whisky", rf"Jack\s+Daniel{Q_}s|Johnnie\s+Walker", "whisky", "m", cat=r"whisk(?:e)?y")
add_special("Martini (vermut) -> vermut",
            rf"\b(?:vermut|vermouth)\s+Martini(?:\s+(?:Rosso|Bianco|Dry|Extra\s+Dry|Rojo|Blanco))?{E_}"
            rf"|{B_}Martini\s+(?:Rosso|Bianco|Extra\s+Dry|Rojo){E_}", "vermut", scope="text",
            trig=r"Martini")

# ---- Chocolates, dulces y botanas ----------------------------------------------------
add_special("Kinder Sorpresa -> huevo de chocolate sorpresa",
            rf"\bhuevos?\s+Kinder(?:\s+sorpresa)?{E_}|{B_}Kinder\s+Sorpresa{E_}", "huevo de chocolate sorpresa", trig=r"Kinder")
add_special("Kinder Bueno -> chocolate con avellanas",
            rf"{B_}Kinder\s+Bueno(?:\s+(?:White|Blanco|Mini))?{E_}", "chocolate con avellanas", trig=r"Kinder")
add_brand("Kinder -> chocolate", r"Kinder(?:\s+(?:Delice|Maxi|Chocolate|Country|Joy|Happy\s+Hippo))?", "chocolate", "m")
add_brand("Kit Kat -> chocolate con barquillo", r"Kit[\s-]?Kats?", "chocolate con barquillo", "m", cat=r"chocolates?")
add_brand("M&M's -> grageas de chocolate de colores", rf"M\s?&\s?M(?:{Q_}s)?", "grageas de chocolate de colores", "f",
          cat=r"chocolates?|grageas?|confites?|dulces?|lunetas?")
add_brand("Snickers -> barra de chocolate con cacahuate", r"Snickers", "barra de chocolate con cacahuate", "f")
add_brand("Twix -> barra de chocolate con galleta", r"Twix", "barra de chocolate con galleta", "f")
add_brand("Toblerone -> chocolate con turrón", r"Toblerone", "chocolate con turrón", "m", cat=r"chocolate")
add_brand("Ferrero Rocher -> bombones de chocolate y avellana", r"Ferrero(?:\s+Rocher)?|Rocher", "bombones de chocolate y avellana", "m",
          cat=r"bombones?|chocolates?")
add_brand("Reese's -> chocolate con crema de cacahuate", rf"Reese(?:{Q_}s)?", "chocolate con crema de cacahuate", "m",
          cat=r"chocolates?|barras?|copas?|(?:crema|mantequilla)\s+de\s+cacahuate")
add_brand("Lindt/Kellogg's -> (eliminada)", rf"Lindt|Kellogg{Q_}s", "", "m")
add_brand("Gansito -> pastelito relleno", r"Gansitos?", "pastelito relleno", "m", cat=r"pastelitos?|pastelillos?")
add_brand("Abuelita/Ibarra/Carlos V -> chocolate de mesa", r"Abuelita|Ibarra|Carlos\s+V", "chocolate de mesa", "m",
          cat=r"chocolates?(?:\s+de\s+mesa)?|tablillas?(?:\s+de\s+chocolate)?")
add_brand("Nesquik -> chocolate en polvo", r"Nesqu[ií]c?k", "chocolate en polvo", "m",
          cat=r"chocolate(?:\s+en\s+polvo)?|cacao(?:\s+en\s+polvo)?|cocoa|polvo\s+(?:para\s+preparar\s+)?(?:de\s+)?chocolate|"
              r"saborizante(?:\s+de\s+chocolate)?|bebida\s+(?:de\s+)?chocolate")
add_special("Nesquik (jarabe) -> jarabe de chocolate",
            rf"\b(?:jarabe|sirope|salsa)(?:\s+de\s+chocolate)?\s+(?:de\s+)?Nesqu[ií]c?k{E_}{M}", "jarabe de chocolate", trig=r"Nesqu")
add_brand("Hershey's Kisses -> chocolates pequeños", rf"Hershey{Q_}s?\s+(?:Kisses|Besos)|Kisses", "chocolates pequeños", "m",
          cat=r"chocolates?|besos")
add_special("Hershey's (jarabe) -> jarabe de chocolate",
            rf"\b(?:jarabe|sirope|salsa)(?:\s+de\s+chocolate)?\s+(?:de\s+)?Hershey{Q_}s?{E_}{M}", "jarabe de chocolate", trig=r"Hershey")
add_special("Hershey's -> chocolate/cocoa genérico",
            rf"\b((?:chocolate|cocoa|cacao)(?:\s+en\s+polvo)?)\s+(?:de\s+|marca\s+)?Hershey{Q_}s?{E_}{M}", r"\1", trig=r"Hershey")
add_brand("Hershey's -> chocolate", rf"Hershey{Q_}s?", "chocolate", "m")
add_brand("Nocilla -> crema de avellanas y cacao", r"Nocilla", "crema de avellanas y cacao", "f",
          cat=r"crema(?:\s+de\s+(?:avellanas?|cacao))*(?:\s+y\s+cacao)?|chocolate\s+para\s+untar")
add_brand("Cola Cao -> cacao en polvo", r"Cola[\s-]?Cao", "cacao en polvo", "m", cat=r"cacao(?:\s+en\s+polvo)?")
add_brand("Doritos -> totopos de maíz", r"Doritos", "totopos de maíz", "m", cat=r"totopos|frituras|botanas?")
add_brand("Cheetos -> frituras de maíz con queso", r"Cheetos", "frituras de maíz con queso", "f", cat=r"frituras|botanas?")
add_brand("Takis -> frituras de maíz enrolladas", r"Takis", "frituras de maíz enrolladas", "f", cat=r"frituras|botanas?")
add_brand("Sabritas/Pringles/Ruffles -> papas fritas", r"Sabritas|Pringles|Ruffles", "papas fritas", "f",
          cat=r"papas(?:\s+fritas)?|botanas?")
add_brand("Tajín -> chile en polvo con limón", r"Taj[ií]n", "chile en polvo con limón", "m",
          cat=r"chile(?:\s+en\s+polvo)?(?:\s+con\s+lim[oó]n)?|sazonador")

# ---- Cereales, galletas y panes ------------------------------------------------------
add_brand("Cheerios -> cereal de avena en aros", r"Cheerios", "cereal de avena en aros", "m", cat=r"cereal(?:\s+de\s+avena)?")
add_brand("Corn Flakes -> hojuelas de maíz", r"Corn[\s-]?Flakes", "hojuelas de maíz", "f", cat=r"cereal(?:\s+de)?|hojuelas|copos")
add_brand("Choco Krispis -> cereal de arroz inflado con chocolate", r"Choco[\s-]?Krispis", "cereal de arroz inflado con chocolate", "m",
          cat=r"cereal")
add_brand("Maseca -> harina de maíz nixtamalizado", r"Maseca", "harina de maíz nixtamalizado", "f",
          cat=r"harina(?:\s+de\s+ma[ií]z)?(?:\s+nixtamalizad[oa])?|masa\s+harina")
add_brand("Mazola -> aceite de maíz", r"Mazola", "aceite de maíz", "m", cat=r"aceite(?:\s+de\s+ma[ií]z)?")
add_brand("Knox -> gelatina sin sabor", r"Knox", "gelatina sin sabor", "f", cat=r"gelatina(?:\s+sin\s+sabor)?|grenetina")
add_brand("Carnation -> leche evaporada", r"Carnation", "leche evaporada", "f", cat=r"leche(?:\s+evaporada)?")
add_brand("Lyncott -> queso crema", r"Lyncott", "queso crema", "m", cat=r"queso(?:\s+crema)?")
add_brand("Alpura -> leche", r"Alpura", "leche", "f", cat=r"leche")
add_brand("Gran Capitán -> queso", r"Gran\s+Capit[aá]n", "queso", "m", cat=r"queso", plain=False)

for _b, _gp, _gs in [
    (r"Ritz(?:\s+(?:Crackers|Bits))?|Ducales|Saladitas|Wheat\s+Thins", "galletas saladas", "galleta salada"),
    (r"Chips\s+Ahoy", "galletas con chispas de chocolate", "galleta con chispas de chocolate"),
    (r"Emperador", "galletas rellenas", "galleta rellena"),
    (r"(?:Lotus\s+)?Biscoff", "galletas de caramelo especiado", "galleta de caramelo especiado"),
]:
    add_special(f"{_gp} (marca eliminada, singular)", rf"\bgalleta\s+(?:(?:tipo|de|marca)\s+)?(?:{_b}){E_}{M}", _gs, trig=_b)
    add_brand(f"-> {_gp}", _b, _gp, "f", cat=r"galletas?", plain=_b != r"Emperador")

add_brand("Thins (pan) -> pan delgado", r"(?-i:Thins)", "pan delgado", "m", cat=r"pan")

# ---- Galletas María (tipo de galleta, no se tocan 'baño María' ni nombres propios) ------
add_special("Galletas María -> galletas dulces",
            rf"\bgalletas\s+(?:(?:tipo|de\s+la\s+marca|de|marca)\s+)?Mar[ií]as?{E_}", "galletas dulces", trig=r"galletas?\s+(?:\w+\s+)*Mar[ií]a")
add_special("Galleta María -> galleta dulce",
            rf"\bgalleta\s+(?:(?:tipo|de\s+la\s+marca|de|marca)\s+)?Mar[ií]as?{E_}", "galleta dulce", trig=r"galleta\s+Mar[ií]a")
add_special("'Digestive/Chiquilín/galletas dulces o María' -> sin María",
            rf"\b(Digestive|Chiquil[ií]n|galletas?\s+(?:dulces?|saladas?))\s*(?:o|y|/)\s*Mar[ií]as?{E_}", r"\1", trig=r"Mar[ií]as?")
add_special("'galletas ..., tipo María' -> galletas ...",
            rf"(\bgalletas?\b[^|()]{{0,30}}?)[,\s]+(?:tipo|estilo)\s+Mar[ií]as?{E_}", r"\1", trig=r"(?:tipo|estilo)\s+Mar[ií]a")
add_special("'tipo María' -> tipo galleta dulce",
            rf"\b(tipo|estilo)\s+Mar[ií]as?{E_}", r"\1 galleta dulce", trig=r"(?:tipo|estilo)\s+Mar[ií]a")
add_special("Marías (plural) -> galletas dulces",
            rf"(?<!Tres )(?<!Ave )(?<!Santa )\bMar[ií]as{E_}", "galletas dulces", trig=r"Mar[ií]as")

# ---- Salsas y sazonadores ------------------------------------------------------------
add_special("'salsa tipo Tabasco' -> salsa picante",
            rf"\bsalsa\s+(?:picante\s+)?(?:tipo|estilo|como)\s+Tabasco{E_}{M}", "salsa picante", trig=r"Tabasco")
add_special("'ají/chile Tabasco' -> ají/chile picante",
            rf"\b(aj[ií]|chile)\s+Tabasco{E_}{M}", r"\1 picante", scope="text", trig=r"Tabasco")
add_special("'salsa (de) Tabasco' -> salsa picante",
            rf"\bsalsa(?:\s+picante)?(?:\s+de)?\s+Tabasco{E_}{M}", "salsa picante", trig=r"Tabasco")
add_special("', tipo Tabasco' -> eliminado",
            rf",?\s*\b(?:tipo|estilo|como)\s+Tabasco{E_}{M}", "", scope="text", trig=r"Tabasco")
add_special("'el/del Tabasco' -> la/de la salsa picante",
            rf"\b(el|del)\s+Tabasco{E_}{M}", lambda m: {"el": "la", "del": "de la"}[m.group(1).lower()] + " salsa picante",
            scope="text", trig=r"Tabasco")
add_special("Tabasco -> salsa picante (solo texto; en títulos 'con Tabasco')",
            rf"(?<!estado de )(?<!regi[oó]n de )(?<!ciudad de )(?<!chocolate de )(?<!cacao de ){B_}Tabascos?{E_}{M}",
            "salsa picante", scope="text", trig=r"Tabasco")
add_special("Tabasco -> salsa picante (títulos)", rf"(?<=\bcon ){B_}Tabasco{E_}{M}", "salsa picante", scope="name", trig=r"Tabasco")
add_brand("Sriracha -> salsa picante", r"Sriracha", "salsa picante", "f", cat=r"salsa(?:\s+picante)?(?:\s+de\s+chile)?")
add_brand("Cholula/Tapatío/Frank's RedHot -> salsa picante", rf"Cholula|Tapat[ií]o|Frank{Q_}s\s+Red\s?Hot|Red\s?Hot", "salsa picante", "f",
          cat=r"salsa(?:\s+picante)?")
add_special("Valentina -> salsa picante",
            rf"\bsalsa(?:\s+picante)?\s+(?:de\s+)?Valentina{E_}"
            rf"|(?<=Catsup y )Valentina{E_}|(?<=Ketchup o )Valentina{E_}|(?<=como )valentina{E_}"
            rf"|(?<=salsa roja, )valentina{E_}|(?<=picante, )valentina{E_}", "salsa picante", trig=r"valentina")
add_special("maggie (minúscula, salsa sazonadora) -> salsa sazonadora",
            rf"{B_}(?-i:maggie){E_}", "salsa sazonadora", flags=re.I, trig=r"maggie")
add_special("'Mis Recetas Al Horno' (sazonador) -> eliminado",
            r"\bMis\s+Recetas\s+(?=Al\s+(?:Horno|Tomate))", "", scope="text", trig=r"Mis\s+Recetas")
add_special("Orlando (tomate frito) -> tomate frito",
            rf"\b(tomate\s+frito|salsa\s+de\s+tomate|tomates?\s+fritos?)\s+Orlando(?:\s+(?:Casero|Cl[aá]sic[oa]))?{E_}", r"\1",
            scope="text", trig=r"Orlando")
add_special("Orlando (tomate) -> tomate frito",
            rf"\b(tomates?)\s+Orlando(?:\s+(?:Casero|Cl[aá]sic[oa]))?{E_}", "tomate frito", scope="text", trig=r"Orlando")

# ---- Milo, La Lechera y Royal (los casos que dejaba la lista original) -------------------
add_special("Galletas Milo -> galletas de malta achocolatada",
            rf"\bgalletas?\s+(?-i:Milo){E_}", "galletas de malta achocolatada", trig=r"(?-i:Milo)")
add_brand("Milo -> malta achocolatada", r"(?-i:Milo|MILO)", "malta achocolatada", "f",
          cat=r"malta(?:\s+achocolatada)?|chocolate\s+(?:en\s+polvo|instant[aá]neo)|cacao\s+en\s+polvo|cocoa|"
              r"polvo\s+de\s+chocolate|bebida\s+(?:de\s+)?chocolate")
add_special("milo (minúscula, en medida) -> malta achocolatada",
            r"\b((?:tazas?|cucharad(?:a|ita)s?|gramos?)\s+(?:soperas?\s+)?de\s+(?:chocolate,\s+)?)milo\b", r"\1malta achocolatada",
            trig=r"\bmilo\b")
add_special("el milo -> la malta achocolatada",
            r"\b(?:y\s+)?\bel\s+milo\b", lambda m: m.group(0)[:-len("el milo")] + "la malta achocolatada", trig=r"\bel\s+milo\b")
add_special("alimento en polvo milo -> malta achocolatada",
            r"\balimento\s+en\s+polvo\s+milo\b", "malta achocolatada", trig=r"\bmilo\b")
add_special("chocolate, milo -> chocolate, malta achocolatada",
            r"\bchocolate,\s+milo\b", "chocolate, malta achocolatada", trig=r"\bmilo\b")
add_special("Lechera -> leche condensada",
            rf"\b(de|con|y|o|la|una)\s+Lechera{E_}{M}", r"\1 leche condensada", trig=r"Lechera")
add_special("Lechera (suelta) -> leche condensada",
            rf"(?<!olla )(?<!jarra )(?<!c[aá]ntaro )(?<!bid[oó]n ){B_}Lechera{E_}{M}", "leche condensada", trig=r"Lechera")
add_special("Royal + gelatina/flan -> gelatina/flan",
            rf"\b(gelatina|flan|natilla|pud[ií]n|bud[ií]n|postre)\s+(?:marca\s+|de\s+)?Royal{E_}(?!\s+icing)", r"\1", trig=r"Royal")
add_special("Royal + polvo/levadura -> polvo/levadura",
            rf"\b(polvo\s+(?:para|de)\s+hornear|levadura(?:\s+qu[ií]mica)?)\s*[,(]?\s*(?:tipo\s+|marca\s+|de\s+)?Royale?{E_}(?!\s+icing)\)?",
            r"\1", trig=r"Royale?")
add_special("'la Royal' -> el polvo para hornear",
            rf"\bla\s+Royal{E_}(?!\s+(?:icing|jelly|gala|canin|oak|prestige))", "el polvo para hornear", trig=r"Royal")
add_special("royal (minúscula, sin 'icing') -> polvo para hornear",
            rf"(?<!kir ){B_}royal{E_}(?!\s+(?:icing|jelly|gala|canin|oak|prestige|wedding|blue|flush|family))", "polvo para hornear",
            trig=r"royal")

# Una sola regex que dice "este texto tiene alguna marca" (así no corremos ~150 reglas en textos sin marcas).
TRIGGER_RX = re.compile("|".join(f"(?:{t})" for t in _TRIG), re.I)
# Evita repeticiones como "galletas saladas, galletas saladas" después de reemplazar.
DUP_RX = re.compile(
    r"\b(" + "|".join(re.escape(g) for g in sorted(_GENERICS, key=len, reverse=True)) + r")(?:\s*(?:,|/|\bo\b|\by\b|\bu\b)\s*\1)+\b", re.I)


# Pone mayúsculas tipo título al reemplazo si el original venía en título.
def _title_phrase(s, at_start):
    out = []
    for i, w in enumerate(s.split(" ")):
        out.append(w if (w in _SMALL_T and (i or not at_start) ) else w[:1].upper() + w[1:])
    return " ".join(out)


# Hace que el reemplazo copie el formato del original (MAYÚSCULAS, Título o minúscula).
def _shape(rep, is_name):
    """Adapta el reemplazo al formato del texto original (MAYÚSCULAS, Título, minúscula)."""
    def f(m):
        r = rep(m) if callable(rep) else m.expand(rep)
        if not r:
            return r
        txt = m.group(0)
        letters = [c for c in txt if c.isalpha()]
        if len(letters) > 3 and all(c.isupper() for c in letters):
            return r.upper()
        words = [w for w in txt.split() if w[:1].isalpha()]
        if is_name or (len(words) > 1 and all(w[0].isupper() for w in words)):
            return _title_phrase(r, m.start() == 0)
        return r
    return f


# Aplica la lista ampliada de marcas a un texto.
def clean_brands_extra(t, is_name=False):
    if not TRIGGER_RX.search(t):
        return t
    for label, rx, rep, scope in EXTRA:
        if (scope == "text" and is_name) or (scope == "name" and not is_name):
            continue
        t = rsub(rx, _shape(rep, is_name), t, "1", label)
    return DUP_RX.sub(lambda m: m.group(1), t)

In [ ]:
# Genéricos que, por venir de un reemplazo de marca, pueden quedar duplicados en un renglón.
_DUP_GENERIC = re.compile(
    r"\b(queso crema|caldo en polvo|leche condensada|polvo para hornear|salsa picante|"
    r"fécula de maíz|refresco de cola|crema de avellanas y cacao|robot de cocina|leche evaporada)\b"
    r"((?:\s+\w+){0,2}?)\s*(?:,|/|\bo\b|\bu\b|\by\b)?\s*\1\b", re.I)


# FUNCIÓN PRINCIPAL DE MARCAS: aplica todas las reglas (1a..1f) a un texto.
# is_name=True cuando el texto es un título (ahí las reglas son un poco distintas).
def clean_brands(t, is_name=False):
    o = t
    if is_name:     # "Thermomix Paella" -> "Paella" ; "Paella (Thermomix)" -> "Paella"
        t = rsub(re.compile(rf"^\s*{THERMO}{M}\s*[-–:]?\s+", re.I), "", t, "1", "Thermomix al inicio del título removido")
        t = rsub(re.compile(rf"\s*[-–:(]?\s*(?<!con )(?<!en )(?<!para )(?<!de ){THERMO}{M}\)?\s*$", re.I), "", t, "1", "Thermomix al final del título removido")
        t = rsub(RX_KNORR_NAME, "", t, "1", "Knorr/Maggi/Avecrem removido del título")
    # 1a) marcas ambiguas, solo si llevan ® / ™
    for rx, rep in RX_WITH_MARK_REPL:
        t = rsub(rx, rep, t, "1", "marca con ®/™ -> genérico")
    t = rsub(RX_WITH_MARK, "", t, "1", "marca ambigua + ®/™ eliminada")
    # 1b) reemplazos genéricos
    for label, rx, rep in REPLACEMENTS:
        t = rsub(rx, _shape(rep, is_name), t, "1", label)
    # 1b-bis) lista ampliada de marcas (bebidas, licores, chocolates, galletas, salsas...)
    t = clean_brands_extra(t, is_name)
    # 1c) eliminación de marcas
    for label, rx in ALWAYS_RULES:
        t = rsub(rx, "", t, "1", label)
    # 1d) auditoría de marcas no cubiertas (aún con ®/™)
    for m in RX_RESIDUAL.finditer(t):
        k = m.group(1).strip()
        AUDIT[k] += 1
        AUDIT_EX.setdefault(k, t[max(0, m.start() - 40):m.end() + 30])
    # 1e) marcas no listadas: token capitalizado inmediatamente antes de ®/™
    if AUTO_STRIP:
        t = rsub(RX_AUTO, "", t, "1", "marca no listada (token previo a ®/™) eliminada automáticamente")
    # 1f) símbolos ® ™ y TM suelto
    t, n = re.subn(r"\s?[®™]", "", t)
    if n:
        STATS[("1", "símbolos ®/™ eliminados")] += n
    t, n = re.subn(r"(?<=[a-záéíóúñ0-9])TM\b|\(?\bTM\s?-?\s?\d{1,2}\b\)?", "", t)
    if n:
        STATS[("1", "'TM' / modelo TM5 eliminado")] += n
    # Al cambiar una marca por su genérico, a veces el genérico queda dos veces en el mismo
    # renglón ("queso crema fresco queso crema"). Dejamos solo la primera aparición.
    t = _DUP_GENERIC.sub(r"\1\2", t)
    t = re.sub(r"\bpolvo (?:para|de) hornear\b\s*(?:,|/|\bo\b|\bu\b)\s*polvo (?:para|de) hornear\b",
               "polvo para hornear", t, flags=re.I)
    t = re.sub(r"\b(queso crema)(?:\s+(?:de\s+)?queso crema)+", r"\1", t, flags=re.I)
    t = re.sub(r"\b(recipiente de silicón)(?:\s+(?:de\s+)?\1)+", r"\1", t, flags=re.I)
    t = re.sub(r"\b(edulcorante|endulzante)\s+(?:o|y|/)\s+\1\b", r"\1", t, flags=re.I)
    return tidy(t) if t != o else t

In [ ]:
# Cómo se escribe Kiwilimón (con o sin tilde, junto o separado).
KIWI = r"kiwi\s?lim[oó]n"
# Frases que mencionan a Kiwilimón y se eliminan.
K_RULES = [
    ("URL de kiwilimon", re.compile(r"\(?\s*(?:https?://|www\.)\S*kiwilimon\S*\s*\)?", re.I)),
    ("paréntesis con 'kiwilimon'", re.compile(rf"\s*\([^()|]*{KIWI}[^()|]*\)", re.I)),
    ("'ver receta ... en kiwilimon'",
     re.compile(rf",?\s*\bver\s+(?:la\s+)?receta[^|.;()]{{0,80}}?(?:en|de|del)\s+{KIWI}", re.I)),
    ("'receta de ... en kiwilimon'",
     re.compile(rf"\s*(?:\b(?:con|seg[uú]n|de)\s+)?(?:la\s+|esta\s+)?\breceta\s+(?:de\s+|del\s+|que\s+est[aá]\s+)?"
                rf"[^|.;()]{{0,80}}?\s*(?:en|de|del)\s+{KIWI}", re.I)),
    ("'los clásicos de kiwilimón'",
     re.compile(rf"[¡!]?\s*[^|.!¡?]*cl[aá]sicos\s+de\s+{KIWI}\s*[!.]?", re.I)),
]
# Si todavía queda alguna mención, se quita la oración completa.
K_FALLBACK = re.compile(rf"[^|.!?]*{KIWI}[^|.!?]*[.!?]?\s*", re.I)


def _prev_word(s, pos):
    w = re.findall(r"(\w+)\s*$", s[:pos])
    return w[0].lower() if w else ""


# En títulos: "Kiwilimón" se quita, pero "Mojito de Kiwi Limón" (la fruta) se respeta.
def _kiwi_name(m):
    if m.group(1):                      # "Kiwilimón" en una sola palabra
        return ""
    if _prev_word(m.string, m.start()) in {"de", "con", "y", "e", "o", "al", "el", "la", "un", "una", "del", "jugo", "sabor"}:
        return m.group(0)               # "Mojito de Kiwi Limón" = fruta kiwi + limón
    return ""


# PASO 4 para títulos.
def clean_kiwi_name(t):
    o = t
    rx = re.compile(r"\s*[-–—|:]?\s*(?:\b(Kiwilim[oó]n)\b|\bKiwi\s+Lim[oó]n\b)", re.I)
    t = rsub(rx, _kiwi_name, t, "2", "'Kiwilimón' removido del título")
    return tidy(t) if t != o else t


# PASO 4 para ingredientes y pasos.
def clean_kiwi_text(t, numbered=False):
    o = t
    if not re.search(KIWI, t, re.I):
        return t
    for label, rx in K_RULES:
        t = rsub(rx, "", t, "2", label)
    if re.search(KIWI, t, re.I):                       # mención residual -> se quita la oración
        segs = []
        for seg in t.split(" | "):
            seg, n = K_FALLBACK.subn("", seg)
            if n:
                STATS[("2", "oración residual con 'kiwilimon' eliminada")] += n
            segs.append(seg)
        t = " | ".join(segs)
    t = tidy(t)
    return renumber(t) if (numbered and " | " in o and t.count(" | ") != o.count(" | ")) else t


# ----------------------------------------------------------------------------
# PASO 6: notas de porciones dentro de la lista de ingredientes
# ----------------------------------------------------------------------------
# Renglones como "Para 4 personas" o "Rinde 6 porciones": no son un ingrediente,
# son el rendimiento de la receta (el dato que decidimos no usar).
SERVINGS_LINE = re.compile(
    r"^\s*(?:para|rinde|salen?|rendimiento)\s*:?\s*\d+\s*(?:[-–aoy/]\s*\d+)?\s*"
    r"(?:personas?|comensales|raciones?|porciones?)\s*\.?\s*$", re.I)


def drop_servings_lines(t):
    """Quita de la lista de ingredientes los renglones que solo dicen para cuántos rinde."""
    partes = t.split(" | ")
    buenas = [p for p in partes if not SERVINGS_LINE.match(p)]
    if len(buenas) != len(partes):
        STATS[("7", "renglón de porciones eliminado de los ingredientes")] += len(partes) - len(buenas)
    return " | ".join(buenas)


# ----------------------------------------------------------------------------
# PASO 5: autopromoción y enlaces
# ----------------------------------------------------------------------------
# Detecta enlaces y direcciones web.
URL_RX = re.compile(r"(?:(?:Imagen|Foto|Truco|V[ií]deo|Video|Fuente)\s*:\s*)?"
                    r"(?:https?://\S+|\bwww\.\S+|\b[\w-]+\.(?:com|net|org|es|mx|blogspot\.com|wordpress\.com)\b(?:/\S*)?)", re.I)
# Palabras de "llamada a la acción" (visita, mira, sigue...).
CTA_RX = re.compile(r"consulta|visita|m[aá]s recetas|mira|descubre|encuentra|sigue|v[ií]deo|video|canal|blog|"
                    r"pincha|haz clic|click|entra", re.I)
# Frases de autopromoción (mi blog, sígueme, suscríbete, redes sociales...).
PROMO_RX = re.compile(
    r"\b(?:mi|mis|nuestro|nuestra|nuestros|nuestras)\s+(?:blogs?|canal|perfil|p[aá]gina|web|sitio|youtube|instagram|facebook|tiktok|redes|recetario)\b"
    r"|\bs[ií]gu(?:enos|eme|enme|anos)\b|\bsuscr[ií]b(?:e|ete|ir|irte|ase|anse)\b|\bcampanita\b"
    r"|\b(?:instagram|facebook|tiktok|pinterest|telegram|whatsapp|twitter|youtube)\b"
    r"|\bvisit(?:a|ar|es|e|ando)\s+(?:\w+\s+){0,2}?(?:blog|canal|web|p[aá]gina|perfil|sitio)\b"
    r"|\bvisit(?:a|ar|es|e|ando)\s+(?-i:[A-ZÁÉÍÓÚÑ])"
    r"|[\w.+-]+@[\w-]+\.\w+|(?<![\w])@\w{3,}"
    r"|\bp[aá]sate\s+por\b|\becha\s+un\s+vistazo\s+a\s+mi\b"
    r"|\b(?:mira|descubre|ve|ver)\s+(?:aqu[ií]\s+)?el\s+v[ií]deo\b|\bv[ií]deo\s+aqu[ií]\b"
    r"|¿te\s+interesan\s+m[aá]s\s+recetas|\bsi\s+te\s+ha\s+gustado\s+esta\s+receta\b", re.I)
# Paréntesis promocionales, por ejemplo "(receta en mi perfil)".
PROMO_PAREN = re.compile(r"\s*\([^()|]*(?:\b(?:mi|nuestro)\s+(?:perfil|blogs?|canal|recetario)|instagram|youtube|facebook|tiktok)[^()|]*\)", re.I)
SENT_SPLIT = re.compile(r"(?<=[.!?])\s*(?=[A-ZÁÉÍÓÚÑ¡¿])|(?<=[.!?])\s+")
CLAUSE_SPLIT = re.compile(r"(?<=[,;])\s+")


# PASO 5: borra oraciones con enlaces o autopromoción.
def clean_promo(t, numbered=False):
    o = t
    if not (URL_RX.search(t) or PROMO_RX.search(t)):
        return t
    t, n = PROMO_PAREN.subn("", t)
    if n:
        STATS[("6", "paréntesis promocional eliminado (ej. '(receta en mi perfil)')")] += n
    segs, dropped = [], 0
    for seg in t.split(" | "):
        keep = []
        if not numbered:        # ingredientes: se quita solo la cláusula promocional, no el ingrediente
            parts = CLAUSE_SPLIT.split(seg)
            if len(parts) > 1 and not PROMO_RX.search(parts[0]) and not URL_RX.search(parts[0]):
                good = [p_ for p_ in parts if not PROMO_RX.search(p_) and not (URL_RX.search(p_) and CTA_RX.search(p_))]
                if len(good) < len(parts):
                    dropped += 1
                    STATS[("6", "cláusula promocional removida de un ingrediente")] += 1
                    seg = " ".join(good)
        for sent in SENT_SPLIT.split(seg):
            if URL_RX.search(sent):
                if CTA_RX.search(sent) or PROMO_RX.search(sent):
                    dropped += 1
                    STATS[("6", "oración con enlace + llamada a la acción")] += 1
                    continue
                sent = URL_RX.sub("", sent)
                STATS[("6", "URL suelta eliminada")] += 1
            elif PROMO_RX.search(sent):
                dropped += 1
                STATS[("6", "oración autopromocional eliminada")] += 1
                continue
            keep.append(sent)
        seg2 = " ".join(keep).strip()
        if re.fullmatch(r"\d*\W*", seg2):
            seg2 = ""
        if seg2:
            segs.append(seg2)
    t = tidy(" | ".join(segs))
    return renumber(t) if (numbered and dropped and " | " in o) else t

In [ ]:
df = recipes.copy()
text_before = df[["name", "ingredients", "instructions"]].copy()    # para comparar marcas al final

STATS.clear(); AUDIT.clear(); AUDIT_EX.clear(); CHANGES.clear()

S1, S2, S3, S4, S5, S6 = ("1. Listas de Python -> texto", "2. Letras rotas, emojis y HTML",
                          "3. Marcas comerciales", "4. Menciones a Kiwilimón", "5. Autopromoción y enlaces",
                          "6. Notas de porciones en los ingredientes")

apply_step(df, S1, ["ingredients"], desparse_ingredients)
apply_step(df, S1, ["instructions"], desparse_instructions)

apply_step(df, S2, ["name", "ingredients", "instructions"], fix_encoding)
apply_step(df, S2, ["ingredients", "instructions"], fix_angle)

apply_step(df, S3, ["name"], lambda t: clean_brands(t, True))
apply_step(df, S3, ["ingredients", "instructions"], clean_brands)

apply_step(df, S4, ["name"], clean_kiwi_name)
apply_step(df, S4, ["ingredients"], clean_kiwi_text)
apply_step(df, S4, ["instructions"], lambda t: clean_kiwi_text(t, True))
apply_step(df, S4, ["meal_type", "category_tags"],
           lambda t: tidy(re.sub(rf"\s*\b{KIWI}\b", "", t, flags=re.I)) if re.search(KIWI, t, re.I) else t)

apply_step(df, S5, ["ingredients"], clean_promo)
apply_step(df, S5, ["instructions"], lambda t: clean_promo(t, True))

apply_step(df, S6, ["ingredients"], drop_servings_lines)

In [ ]:
empty_name = df["name"].fillna("").str.strip().eq("")
empty_ingredients = df["ingredients"].fillna("").str.strip().eq("")
df = df.loc[~empty_name & ~empty_ingredients].copy()
log_rows("Después de la limpieza profunda", len(df))

## 7. Pasos simples (cada paso, una instrucción) y recetas sin pasos

In [ ]:
# Abreviaturas que terminan en punto pero NO terminan una oración.
ABBREVIATIONS = {"gr", "grs", "kg", "min", "mins", "seg", "ml", "lt", "lts", "cda", "cdas", "cdta", "cdtas",
                 "cdita", "cditas", "aprox", "ej", "etc", "sra", "sr", "dr", "dra", "vs", "oz", "lb", "cc",
                 "máx", "mín", "núm", "no", "ud", "uds"}

SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÑ¡¿])")       # fin de oración + mayúscula
GLUED_SENTENCES = re.compile(r"(?<=[a-záéíóúñ]{3})\.(?=[A-ZÁÉÍÓÚÑ¡¿])")   # "juliana.También"
LEADING_NUMBER = re.compile(r"^(\d+)[.)]?\s+")
LEADING_STEP_LABEL = re.compile(r"^\s*paso\s*\d+\s*[.:\-)]?\s*", re.I)

# Oraciones que NO son pasos:
TIP_START = re.compile(r"^\s*[¡(]?\s*(?:truco|trucos|consejo|consejos|nota|notas|tip|tips|sugerencia|"
                       r"recomendaci[oó]n|dato curioso|ojo|importante)\b\s*[:.\-)]?\s*", re.I)
HAS_TEMPERATURE = re.compile(r"\d\s*[°º]|\bgrados\b|\bprecalient", re.I)
# Frases de despedida: suenan bonito, pero no dicen qué hacer. El `.{0,30}$` del final evita
# borrar un paso de verdad que empiece parecido ("Listo para servir en platos hondos con...").
CLOSING_PHRASE = re.compile(
    r"^[¡!¿?]?\s*(?:"
    r"buen provecho|buen apetito|bon app[eé]tit|a cocinar|a disfrutar|a comer|"
    r"qu?e aproveche|disfruta(?:lo|la|los|las)?|[eé]chale un vistazo|"
    r"espero\s+(?:que\s+)?(?:os|te|les|le|lo|la)?\s*(?:guste|gusten|disfrut|les guste)|"
    r"(?:y\s+)?ya\s+(?:est[aá]n?|estar[aá]n?|estar[ií]an?)\s+list|"
    r"list[oa]s?\s+para\s+(?:comer|servir|disfrutar|degustar|saborear|probar)|"
    r"te\s+encantar[aá]|f[aá]cil,?\s*[¿?]*\s*verdad|verdad\s+que\s+(?:es|fue|era)\s+f[aá]cil|"
    r"eso\s+es\s+todo|delicios[oa]s?|riqu[ií]sim[oa]s?|exquisit[oa]s?"
    r")\b.{0,30}$", re.I)
# Notas del autor al lector ("Espero que os guste...", "Espero les guste esta receta, besos").
# Van aparte de CLOSING_PHRASE porque pueden ser largas: no se les pone límite de letras.
AUTHOR_NOTE = re.compile(r"^[¡!]?\s*espero\s+(?:que\s+)?(?:os|te|les|le|lo|la|nos|se)\b", re.I)
ASK_FOR_FEEDBACK = re.compile(
    r"\bcomentarios?\b|cu[eé]ntanos|d[eé]janos|etiqu[eé]tanos|escr[ií]banos|escr[ií]benos|env[ií]anos|"
    r"comparte(?:nos)?\s+(?:con\s+nosotros|tu|la\s+foto|una\s+foto)|foto(?:graf[ií]a)?\s+del\s+resultado|"
    r"esperamos\s+vuestr", re.I)
SERVINGS_NOTE = re.compile(r"^\s*(?:para|rinde|salen?)\s+\d+\s*(?:personas|porciones|raciones|comensales)\.?\s*$", re.I)
OTHER_RECIPE_CTA = re.compile(r"\b(?:revisa|revisar|consulta|consultar|echa un vistazo|mira|ve la|"
                              r"puedes hacer tambi[eé]n|te recomendamos ver|te ense[ñn]amos|no te pierdas)\b", re.I)


def split_sentences(text):
    """Parte un texto en oraciones sin cortar abreviaturas ni '20 min. Retira'."""
    text = GLUED_SENTENCES.sub(". ", text)
    pieces, start = [], 0
    for m in SENTENCE_BOUNDARY.finditer(text):
        tokens = text[start:m.start()].split()
        last = tokens[-1].lower().lstrip("([¡¿\"'") if tokens else ""
        # "Paso 1." o "1." al inicio es una etiqueta, no el fin de una oración.
        is_number = bool(re.fullmatch(r"\d+[.)]", last)) and (len(tokens) == 1 or
                                                              (len(tokens) == 2 and tokens[0].lower() == "paso"))
        is_abbreviation = last.endswith(".") and last[:-1] in ABBREVIATIONS
        is_initial = re.fullmatch(r"[a-záéíóúñ]\.", last)
        if is_number or is_abbreviation or is_initial:
            continue
        pieces.append(text[start:m.start()].strip())
        start = m.end()
    pieces.append(text[start:].strip())
    return [p for p in pieces if p]


LONG_SENTENCE = 180
LONG_SPLIT_POINTS = re.compile(r";\s+|,\s+(?:y\s+)?(?:luego|despu[eé]s|posteriormente|acto seguido|a continuaci[oó]n)\s+", re.I)


def split_long(sentence):
    """Una oración de más de 180 letras se parte en ';' o en 'luego / después / a continuación'."""
    if len(sentence) <= LONG_SENTENCE:
        return [sentence]
    parts = [p.strip(" ,;") for p in LONG_SPLIT_POINTS.split(sentence)]
    parts = [p[:1].upper() + p[1:] for p in parts if p]
    if len(parts) < 2 or any(len(p) < 25 for p in parts):      # si quedaría un pedazo diminuto, no se parte
        return [sentence]
    return parts


def keep_sentence(sentence):
    """Devuelve la oración (posiblemente sin etiqueta) o None si no es un paso."""
    sentence = LEADING_STEP_LABEL.sub("", sentence)          # "Paso 2 Rellenar..." -> "Rellenar..."
    if not sentence:
        return None
    if re.search(r"\bpaso\s*\d", sentence, re.I):            # "ve al paso 10": esa numeración ya no existe
        return None
    if TIP_START.match(sentence):
        # Un "Truco: precalienta a 180 °C" es una instrucción real: se queda, sin la etiqueta.
        if not HAS_TEMPERATURE.search(sentence):
            return None
        text = TIP_START.sub("", sentence).strip()
        return text[:1].upper() + text[1:]
    if (CLOSING_PHRASE.match(sentence) or AUTHOR_NOTE.match(sentence)
            or ASK_FOR_FEEDBACK.search(sentence) or SERVINGS_NOTE.match(sentence)):
        return None
    if re.search(r"\breceta\b", sentence, re.I) and OTHER_RECIPE_CTA.search(sentence):
        return None
    return sentence


def simplify_steps(text):
    """'1 paso largo con varias oraciones | 2 ...' -> 'oración | oración | oración'."""
    steps = [s.strip() for s in text.split(" | ") if s.strip()]
    steps = [s for s in steps if not re.fullmatch(r"\d*\W*", s)]       # pasos que son solo un número

    # Quitar la numeración solo si es 1, 2, 3... consecutiva (así no se corta "2 tazas de...").
    run = 0
    for i, step in enumerate(steps, start=1):
        m = LEADING_NUMBER.match(step)
        if m and int(m.group(1)) == i:
            run = i
        else:
            break
    if run >= 2:
        steps[:run] = [LEADING_NUMBER.sub("", s, count=1) for s in steps[:run]]

    sentences = []
    for step in steps:
        for sentence in split_sentences(step):
            kept = keep_sentence(sentence)
            if kept:
                sentences.extend(split_long(kept))

    merged = []
    for sentence in sentences:
        if merged and len(sentence) < 12:        # "¡Sirve!" se pega al paso anterior
            merged[-1] += " " + sentence
        else:
            merged.append(sentence)
    return " | ".join(merged)

In [ ]:
def n_steps(text):
    return len(text.split(" | ")) if isinstance(text, str) and text else 0


def avg_step_length(text):
    parts = text.split(" | ") if isinstance(text, str) and text else []
    return float(np.mean([len(p) for p in parts])) if parts else np.nan


antes = df["instructions"].copy()
df["instructions"] = df["instructions"].map(lambda t: simplify_steps(t) if isinstance(t, str) else t)
df["instructions"] = df["instructions"].replace("", np.nan)

sin_pasos = df["instructions"].isna()
df = df.loc[~sin_pasos].copy()
log_rows("Con pasos", len(df))

## 8. Duplicados

In [ ]:
df["name_key"] = df["name"].map(normalize_key)
df["ingredients_key"] = df["ingredients"].map(normalize_key)


def completeness_score(frame):
    # +1 por cada dato opcional que la receta sí tiene; más ingredientes también suma un poco.
    score = frame[["difficulty", "total_time_min", "rating", "meal_type", "category_tags"]].notna().sum(axis=1)
    return score + 0.05 * (frame["ingredients"].str.count(r" \| ") + 1)


df["_score"] = completeness_score(df)
df = df.sort_values(["_score", "rating"], ascending=False)       # lo más completo primero

same_page = df.duplicated(["url", "name_key"], keep="first")
df = df.loc[~same_page]

same_recipe = df.duplicated(["name_key", "ingredients_key"], keep="first")
df = df.loc[~same_recipe]

df = df.drop(columns=["_score", "name_key", "ingredients_key"]).reset_index(drop=True)
log_rows("Sin duplicados", len(df))

## 9. Etiquetas de la app (16 etiquetas fijas)

In [ ]:
ALLOWED_TAGS = ["Desayuno", "Comida", "Cena", "Postre", "Bebida", "Alcohol",
                "Vegetariana", "Cerdo", "Pollo", "Mariscos", "Res", "Vegana",
                "Pavo", "Cordero", "Pasta", "Sopa"]
MEAL_TAGS = {"Desayuno", "Comida", "Cena", "Postre", "Bebida"}
PROTEIN_TAGS = {"Cerdo", "Pollo", "Mariscos", "Res", "Pavo", "Cordero"}
DISH_TAGS = {"Pasta", "Sopa"}


def words_regex(words, extra=""):
    """Crea una regex que busca cualquiera de las palabras (completas, sin importar acentos)."""
    parts = sorted((normalize_key(w) for w in words), key=len, reverse=True)
    pattern = "|".join(re.escape(p) for p in parts)
    if extra:
        pattern += "|" + extra
    return re.compile(rf"\b(?:{pattern})(?:e?s)?\b")        # acepta el plural: "trucha" también encuentra "truchas"


# ---- Cosas que PARECEN carne, lácteo o alcohol pero no lo son (se tapan antes de buscar) ----
NOT_REAL = re.compile("|".join([
    r"(?:caldo|consome|fondo|cubos?)s? (?:de )?(?:verduras?|vegetal(?:es)?) (?:o|u|y) (?:de )?(?:pollo|res|carne|pescado)",   # "caldo vegetal o de pollo"
    r"(?:caldo|consome|fondo|cubos?)s? de (?:pollo|res|carne|pescado) (?:o|u) (?:de )?(?:verduras?|vegetal(?:es)?)",
    r"carne (?:vegetal|vegana|de soya|de soja|de jaca|de seitan|de coco|de membrillo)",
    r"(?:hamburguesa|pollo|chorizo|jamon|tocino|salchicha|atun|bacon|chicharron|nuggets?|queso|mantequilla|crema|leche|yogur|yogurt|mayonesa|huevos?) (?:vegano|vegana|vegetal|vegetales)",
    r"(?:hamburguesa|pollo|chorizo|jamon|tocino|salchicha|atun|bacon|chicharron|nuggets?) de (?:soya|soja|lentejas?|garbanzos?|jaca|seitan|champinones?|setas?)",
    r"queso de (?:nuez|nueces|almendras?|anacardos?|soya|soja|maranon|tofu)",
    r"(?:leche|bebida|yogur|yogurt|helado|crema|mantequilla) de (?:coco|almendras?|soya|soja|avena|arroz|nuez|nueces|anacardos?|maranon|canamo|macadamia|pistaches?|avellanas?|cacahuates?|mani|cacao|karite|ajonjoli|semillas?)",
    r"manteca (?:vegetal|de cacao|de karite|de coco|de cacahuate)",
    r"sustituto de huevo|reemplazo de huevo|huevos? de chocolate|huevos? de pascua|huevos? de gallina|huevos? de codorniz",
    r"miel de (?:maple|agave|maiz|palma|cana|arroz|datil|coco)",
    r"consome (?:de )?(?:verduras?|vegetal|vegetales)|caldo (?:de )?(?:verduras?|vegetal|vegetales)",
    r"sin (?:carne|lacteos|alcohol|atun|pollo|jamon|tocino|pescado|mariscos?|cerdo|res|huevo)",
    r"cerveza de raiz|vino de arroz|licor sin alcohol",
    r"(?:setas?|hongos?|champinones?) ostras?|ostras? (?:setas?|hongos?)",             # el hongo, no el marisco
    r"conejos? de (?:chocolate|pascua|azucar|fondant|mazapan)|galletas? de conejos?|chocolate con conejos?",   # dulces
    r"truchas? de (?:batata|cielo|calabaza|navidad)",                                  # dulce canario
    r"levadura de cerveza|vinagre(?: de)? \w+",
]))

# Caldos y sabores: no son la proteína principal (pero sí hacen la receta NO vegetariana).
FLAVORINGS = re.compile(
    r"\b(?:caldo|consome|cubo|cubos|cubito|cubitos|pastilla|pastillas|sazonador|concentrado|fondo|esencia|sabor|saborizante|"
    r"grasa|manteca|aceite|jugo|salsa|extracto|polvo)\s+(?:en polvo\s+)?(?:de\s+)?"
    r"(?:pollo|res|carne|cerdo|puerco|pescado|marisco|mariscos|camaron|camarones|jamon|tocino|gallina|ostion|ostiones|ostras?)\b"
)

# ---- Palabras que activan cada etiqueta de proteína ----
PROTEINS = {
    "Cerdo": words_regex([
        "cerdo", "puerco", "cochino", "lechón", "tocino", "tocineta", "bacon", "panceta", "jamón", "chorizo",
        "longaniza", "chicharrón", "salami", "pepperoni", "mortadela", "morcilla", "butifarra", "pernil",
        "carnitas", "cochinita", "prosciutto", "al pastor", "lacón", "cochinillo", "sobrasada", "salchichón", "chorizos", "ibérico", "ibérica"]),
    "Pollo": words_regex([
        "pollo", "pechuga", "pechugas", "muslo de pollo", "muslos", "alitas", "gallina", "pollito"]),
    # "carne" sola cuenta como Res, pero "carne de pollo/cerdo/pescado..." no.
    "Res": words_regex([
        "res", "ternera", "bistec", "arrachera", "sirloin", "ribeye", "rib eye", "diezmillo", "suadero", "machaca",
        "cecina", "chambarete", "barbacoa", "solomillo", "puntas de filete", "filete mignon", "t bone",
        "costilla de res", "chuleta de res", "cuete", "bife", "lomo de vaca", "carne de vaca", "buey", "chuletón",
        "rabo de toro", "rabo de res", "bifes", "bistecs", "sobrebarriga"],
        extra=r"carne(?:\s+(?:molida|picada|seca|asada|deshebrada))?"
              r"(?!\s+(?:molida\s+|picada\s+)?de\s+(?:pollo|cerdo|puerco|pescado|cangrejo|jaiba|pavo|cordero|conejo|"
              r"borrego|camaron|marisco|ave|pato|jamon))"),
    "Mariscos": words_regex([
        "pescado", "pescados", "bacalao", "atún", "salmón", "sardina", "sardinas", "anchoa", "anchoas", "trucha",
        "tilapia", "mojarra", "huachinango", "róbalo", "lubina", "merluza", "corvina", "pargo", "mero", "bonito",
        "arenque", "caballa", "surimi", "camarón", "camarones", "langostino", "langostinos", "langosta", "cangrejo",
        "jaiba", "pulpo", "calamar", "calamares", "mejillón", "mejillones", "almeja", "almejas", "ostión",
        "ostiones", "ostra", "ostras", "vieira", "vieiras", "callo de hacha", "abulón", "marisco", "mariscos",
        # Pescados y mariscos que se usan sobre todo en España y que antes pasaban como "vegetarianos".
        "gamba", "gambas", "lenguado", "rape", "boquerón", "boquerones", "cazón", "sepia", "chipirón", "chipirones",
        "besugo", "bogavante", "rodaballo", "abadejo", "berberecho", "berberechos", "congrio", "palometa",
        "pez espada", "centollo", "txangurro", "percebe", "percebes", "salmonete", "salmonetes", "pescadilla",
        "raya", "cigala", "cigalas", "carabinero", "bagre", "erizo de mar", "chirla", "chirlas", "coquina",
        "coquinas", "jurel", "robalo", "choro", "choros", "urta", "lisa a la parrilla", "dorada al horno", "dorada a la sal", "dorada a la plancha"]),
    "Pavo": words_regex(["pavo", "guajolote", "chompipe"]),            # "pechuga de pavo" se cambia a "pavo" más abajo
    "Cordero": words_regex(["cordero", "borrego", "lechal"]),
}

# Otros animales y derivados: no tienen etiqueta propia, pero la receta NO es vegetariana.
OTHER_ANIMAL = words_regex([
    "chivo", "cabrito", "conejo", "venado", "pato", "codorniz", "ganso", "salchicha",
    "salchichas", "hot dog", "hígado", "menudencias", "mondongo", "pancita", "tripas", "costilla", "costillas",
    "chuleta", "chuletas", "lomo", "consomé", "caldo de pollo", "caldo de res", "caldo de pescado",
    "caldo de camarón", "caldo de carne", "salsa de pescado", "salsa de ostión", "salsa de ostras",
    "esencia de pollo", "gelatina de res", "callos", "riñón", "riñones", "sesos", "perdiz", "faisán", "liebre",
    "jabalí", "fumet", "tuétano", "caldo de ave", "caldo de gallina"])

# Si lleva alguno, NO es vegana (aunque sí puede ser vegetariana).
NOT_VEGAN = words_regex([
    "leche", "queso", "quesos", "crema", "nata", "mantequilla", "yogur", "yogurt", "requesón", "ricotta", "mozzarella",
    "mozarela", "parmesano", "cheddar", "panela", "manchego", "cotija", "feta", "gouda", "brie", "mascarpone",
    "gruyère", "huevo", "huevos", "yema", "yemas", "mayonesa", "miel", "lácteos", "lácteo", "suero de leche",
    "dulce de leche", "cajeta", "helado", "gelatina", "grenetina", "colágeno", "manteca", "salsa inglesa",
    "merengue", "chocolate blanco"])

ALCOHOL = words_regex([
    "tequila", "mezcal", "ron", "vodka", "ginebra", "whisky", "whiskey", "brandy", "coñac", "cognac", "cerveza",
    "cervezas", "vino", "vinos", "champagne", "champaña", "prosecco", "cava", "licor", "licores", "amaretto",
    "triple sec", "curaçao", "aperol", "vermut", "vermouth", "jerez", "oporto", "sake", "pisco", "aguardiente",
    "cachaça", "absenta", "pulque", "alcohol", "sangría", "aperitivo amargo"])

DECLARED_VEGAN = re.compile(r"\bveganos?\b|\bveganas?\b")
DECLARED_VEGETARIAN = re.compile(r"\bvegetarianos?\b|\bvegetarianas?\b")


def prepare_for_search(text):
    """Texto en minúsculas y sin acentos, conservando ' | ' entre ingredientes."""
    parts = [normalize_key(p) for p in str(text).split(" | ")]
    text = " | ".join(p for p in parts if p)
    # "pechuga / jamón / salchicha / tocino de pavo" son pavo: no son pollo ni cerdo.
    return re.sub(r"\b(?:pechugas?|jamon(?:es)?|salchichas?|tocino|bacon|chorizo|muslos?|milanesas?)\s+de\s+pavo\b", "pavo", text)


def food_tags(name, ingredients, category_tags):
    """Etiquetas de dieta, proteína y alcohol."""
    name_key = normalize_key(name)
    declared = normalize_key(f"{name} {category_tags}")
    says_vegan = bool(DECLARED_VEGAN.search(declared))
    says_vegetarian = says_vegan or bool(DECLARED_VEGETARIAN.search(declared))
    declared_in_name = bool(DECLARED_VEGAN.search(name_key) or DECLARED_VEGETARIAN.search(name_key))

    ing_real = NOT_REAL.sub(" ", prepare_for_search(ingredients))
    tags = set()

    if ALCOHOL.search(ing_real):
        tags.add("Alcohol")

    both_real = NOT_REAL.sub(" ", prepare_for_search(f"{name} | {ingredients}"))
    for_protein = FLAVORINGS.sub(" ", both_real)

    # Si SOLO la categoría de la fuente dice "vegetariana" pero la receta lleva una proteína de verdad
    # (jamón, atún, pollo...), la categoría se equivocó: no se le cree. Si lo dice el nombre, sí.
    if says_vegetarian and not declared_in_name:
        if any(regex.search(for_protein) for regex in PROTEINS.values()):
            says_vegan = says_vegetarian = False

    if says_vegetarian:
        # La fuente o el nombre lo dicen: se les cree (y no se pone ninguna proteína).
        tags.add("Vegetariana")
        if says_vegan or not NOT_VEGAN.search(ing_real):
            tags.add("Vegana")
        return tags

    for tag, regex in PROTEINS.items():
        if regex.search(for_protein):
            tags.add(tag)

    has_animal = bool(tags) or bool(OTHER_ANIMAL.search(both_real)) or bool(FLAVORINGS.search(both_real))
    if not has_animal:
        tags.add("Vegetariana")
        if not NOT_VEGAN.search(ing_real):
            tags.add("Vegana")
    return tags

In [ ]:
# ---- Momento del día: lo que dice la fuente -> etiquetas de la app ----
# (Las llaves se escriben normales; luego se normalizan para comparar sin acentos.)
MEAL_FROM_SOURCE = {
    # RecetasDeLaAbuela: tipo de plato
    "Desayuno": ["Desayuno"], "Merienda": ["Desayuno"], "Cena": ["Cena"],
    "Postre": ["Postre"], "Cumpleaños": ["Postre"],
    "Bebida": ["Bebida"], "Cóctel": ["Bebida"],
    "Plato principal": ["Comida"], "Acompañamiento": ["Comida"], "Guarnición": ["Comida"],
    "Entrante": ["Comida", "Cena"], "Sopa": ["Comida", "Cena"], "Ensalada": ["Comida", "Cena"],
    # Kiwilimon: primera parte de la URL
    "desayunos": ["Desayuno"], "panes": ["Desayuno"],
    "postres": ["Postre"], "postres para niños": ["Postre"],
    "bebidas": ["Bebida"],
    "platos fuertes": ["Comida"], "pollo y carne": ["Comida"], "pescados y mariscos": ["Comida"],
    "recetas de pastas": ["Comida"], "guarniciones": ["Comida"], "recetas de botanas fáciles": ["Comida"],
    "salsas": ["Comida"], "comida para niños": ["Comida"], "recetas navideñas": ["Comida"],
    "recetas a la parrilla": ["Comida"],
    "ensaladas": ["Comida", "Cena"], "sopas": ["Comida", "Cena"],
}
MEAL_FROM_SOURCE = {normalize_key(k): v for k, v in MEAL_FROM_SOURCE.items()}

# Si la fuente no dice nada, se intenta adivinar por el nombre (el primero que coincida gana).
NAME_RULES = [
    (r"\b(smoothie|licuado|batido|agua de|atole|cafe|café|mojito|coctel|cóctel|cocktail|spritz|horchata|tepache|kombucha|ponche|latte|margarita|sangria|sangría|limonada|naranjada|jugo|aguardiente|licor|daiquiri|trago|infusión|infusion|chocolate caliente|queimada|refresco|colada|malteada|frappe|granizado|cuba libre|caipirinha|mimosa|michelada|sour|tonic|chicha|pisco|sidra|vermut|gin)\b", "Bebida"),
    (r"\b(cupcake|muffin|brownie|galleta|galletas|pastel|pay|flan|helado|mousse|cheesecake|churro|churros|dona|donas|bizcocho|budin|budín|queque|panque|gelatina|sorbete|melcocha|manjar|dulce de|tres leches|mermelada|compota|alfajor|alfajores|cumpleaños|cumpleanos|tarta|tartaleta|postre|natilla|crema catalana|arroz con leche|turrón|turron|mantecado|polvorón|polvoron|trufa|trufas|bombón|bombon|bombones|gomita|gomitas|paleta|nieve|cajeta|dulces|torrija|torrijas|pestiño|buñuelo|tiramisú|tiramisu|panna cotta|coulant|fondant|profiterol|macarons|cocada|rosca|mazapán|mazapan|cuajada|confitura|merengue|pavlova|brazo de gitano|magdalena|magdalenas)\b", "Postre"),
    (r"\b(hotcake|pancake|waffle|omelette|omelet|huevos revueltos|chilaquiles|desayuno|avena|granola|tostada francesa)\b", "Desayuno"),
    (r"\b(taco|enchilada|sopa|caldo|ensalada|pollo|arroz|pasta|pizza|empanada|quesadilla|guiso|lenteja|frijol|carne|pescado|camaron|camarón|lasagna|lasaña|hamburguesa|sandwich|sándwich|ceviche|pozole|mole|burrito|fajita|crema de|salsa|vinagreta|aderezo|dip|ratatouille|risotto|risoto|quinoa|verduras|tamal|tamales|gratinad[oa]s?|papas|patatas|tortilla|esquites|guacamole|hummus|tostada|pickles|escabeche|ensaladilla|croqueta|croquetas|tarta salada|quiche|sopes|gordita|gorditas|nopales|elote|elotes|frijoles|verdura|setas|hongos|champiñones|champinones|calabacitas|berenjena|berenjenas|pimientos|espinacas|garbanzos|lentejas|alubias|judias|judías|coliflor|brocoli|brócoli|rollitos|rollos|brochetas|hamburguesas|bocadillo|bocadillos|torta|chilaquiles|migas|pisto|revuelto|tortilla|pastel de|pastelón|pastelon|empanadilla|empanadillas|canapé|canapes|canapés|tapas|tapa|pinchos|pincho)\b", "Comida"),
]


# ---- Tipo de plato: Pasta y Sopa ----
DISH_FROM_SOURCE = {"Sopa": ["Sopa"], "sopas": ["Sopa"], "recetas de pastas": ["Pasta"]}
DISH_FROM_SOURCE = {normalize_key(k): v for k, v in DISH_FROM_SOURCE.items()}

# Formas de pasta (sin la palabra "pasta" sola, que se trata aparte).
PASTA_SHAPES = words_regex([
    "espagueti", "espaguetis", "spaghetti", "fettuccine", "fetuccini", "tallarín", "tallarines", "macarrón",
    "macarrones", "penne", "fusilli", "rigatoni", "lasaña", "lasañas", "lasagna", "canelón", "canelones", "ravioles",
    "ravioli", "raviolis", "fideo", "fideos", "ñoquis", "gnocchi", "tortellini", "farfalle", "linguine", "coditos"])
# "pasta" sola, pero NO "pasta de ajo / de tomate / de hojaldre / para pizza / frola" (eso no es pasta de comer).
PASTA_WORD = re.compile(r"\bpastas?\b(?!\s+(?:de|para|frola|choux)\b)")

SOUP_WORDS = words_regex(["sopa", "sopas", "caldo", "caldos", "consomé", "pozole", "gazpacho", "menestrone", "minestrone",
                          "ajiaco", "sancocho", "puchero", "chupe", "locro", "bisque", "velouté", "potaje"])
SOUP_COCIDO = re.compile(r"^(?:el mejor )?cocido\b|\bcocido (?:de|a la|madrileno|gallego)\b")   # no "huevo cocido"
# "Crema de ..." es sopa, salvo las cremas dulces o untables.
SWEET_CREAM = words_regex([
    "avellana", "avellanas", "cacao", "chocolate", "mantequilla", "almendra", "almendras", "dulce", "castañas",
    "castaña", "limón", "vainilla", "leche", "queso", "cacahuate", "maní", "pistache", "turrón", "naranja", "fresa",
    "café", "coco", "nuez", "nueces", "ron", "whisky", "yogur", "plátano", "mango", "maracuyá", "fruta", "frutas",
    "caramelo", "galleta", "tres leches"])


def dish_tags(meal_type, category_tags, name, ingredients):
    found = set()
    labels = [meal_type if isinstance(meal_type, str) else ""]
    labels += (category_tags if isinstance(category_tags, str) else "").split(" | ")
    for label in labels:
        found.update(DISH_FROM_SOURCE.get(normalize_key(label), []))

    name_key = normalize_key(name)
    ingredients_key = prepare_for_search(ingredients)
    is_sauce = name_key.startswith("salsa")          # "Salsa para pasta" no es una pasta
    if not is_sauce:
        for text in (name_key, ingredients_key):
            if PASTA_SHAPES.search(text) or PASTA_WORD.search(text):
                found.add("Pasta")

    if SOUP_WORDS.search(name_key) or SOUP_COCIDO.search(name_key):
        found.add("Sopa")
    cream = re.match(r"crema de (.+)", name_key)
    if cream and not SWEET_CREAM.search(cream.group(1)):
        found.add("Sopa")
    return found


def meal_tags(meal_type, category_tags, name, has_main_dish):
    found = set()
    labels = [meal_type if isinstance(meal_type, str) else ""]
    labels += (category_tags if isinstance(category_tags, str) else "").split(" | ")
    for label in labels:
        found.update(MEAL_FROM_SOURCE.get(normalize_key(label), []))

    # "cena" escrita en el tipo, la categoría o el nombre.
    if re.search(r"\bcena\b", normalize_key(f"{labels[0]} {' '.join(labels[1:])} {name}")):
        found.add("Cena")

    if not found:
        # Se busca el nombre tal cual y también en singular ("tacos" -> "taco").
        name_text = name.lower()
        name_text += " " + " ".join(w[:-1] if len(w) > 3 and w.endswith("s") else w for w in name_text.split())
        for pattern, tag in NAME_RULES:
            if re.search(pattern, name_text, flags=re.I):
                # "Pastel de pollo" no es postre ni "Coctel de langostinos" una bebida:
                # si lleva proteína, o es pasta o sopa, esas reglas se saltan.
                if tag in {"Postre", "Desayuno", "Bebida"} and has_main_dish:
                    continue
                found.add(tag)
                break
    return found


def build_tags(row):
    category_tags = row["category_tags"] if isinstance(row["category_tags"], str) else ""
    tags = food_tags(row["name"], row["ingredients"], category_tags)
    tags |= dish_tags(row["meal_type"], category_tags, row["name"], row["ingredients"])
    tags |= meal_tags(row["meal_type"], category_tags, row["name"], bool(tags & (PROTEIN_TAGS | DISH_TAGS)))

    if tags & {"Postre", "Bebida"}:
        tags.discard("Sopa")          # "sopa de leche" dulce: no es una sopa de comer
    if "Sopa" in tags:
        tags |= {"Comida", "Cena"}    # igual que las sopas que ya vienen así en la fuente
    if not (tags & MEAL_TAGS) and (tags & (PROTEIN_TAGS | {"Pasta"})):
        tags.add("Comida")            # con proteína o pasta y sin momento del día: es una comida

    # "Alcohol" solo si es bebida o postre, o si el nombre lo dice ("Pollo al vino").
    # Un chorrito de vino dentro de un guiso no cuenta.
    if "Alcohol" in tags and not (tags & {"Bebida", "Postre"} or ALCOHOL.search(normalize_key(row["name"]))):
        tags.discard("Alcohol")
    assert tags <= set(ALLOWED_TAGS), tags
    return " | ".join(t for t in ALLOWED_TAGS if t in tags) or np.nan

In [ ]:
df["tags"] = df.apply(build_tags, axis=1)

## 10. Dataset final, validación y guardado

In [ ]:
FINAL_COLUMNS = ["recipe_id", "name", "ingredients", "instructions", "tags", "difficulty",
                 "total_time_min", "country", "rating", "source", "url"]

recipes_final = df.reset_index(drop=True)
recipes_final["recipe_id"] = np.arange(1, len(recipes_final) + 1)

# ---- Últimos retoques de Yanuq ----
# 1) Algunos títulos traen el nombre de la página ("Pisco Sour Yanuq", "... al Estilo Yanuq").
recipes_final["name"] = (recipes_final["name"]
                         .str.replace(r"\s*(?:\b(?:al\s+)?estilo\s+)?\byanuq\b", "", case=False, regex=True)
                         .str.strip())

# 2) Al final de los pasos de Yanuq vienen cosas que no son pasos: "12 personas", datos de calorías, "Maridaje"
#    y recomendaciones de otras recetas ("YANUQ SUGIERE PARA ACOMPAÑAR..."). Se corta desde ahí.
YANUQ_TAIL = re.compile(
    r"(?:(?:(?<=[.!?])\s*|^)\d+(?:\s*[-–]\s*\d+)?\s+personas?\b|\bcalor[ií]as\s+\d|\bmaridaje\b|\byanuq\b).*$",
    re.I)


def clean_yanuq_steps(text):
    steps = [YANUQ_TAIL.sub("", step).strip() for step in text.split(" | ")]
    return " | ".join(step for step in steps if step)


is_yanuq = recipes_final["source"].eq("Yanuq")
recipes_final.loc[is_yanuq, "instructions"] = recipes_final.loc[is_yanuq, "instructions"].map(clean_yanuq_steps)
recipes_final = recipes_final[recipes_final["instructions"].str.strip().ne("")]      # por si alguna quedó sin pasos

# Al quitar "Yanuq" del título, "Pisco Sour Yanuq" puede quedar igual que "Pisco Sour": es la misma receta repetida.
same_page_and_name = recipes_final.assign(name_key=recipes_final["name"].map(normalize_key)).duplicated(["url", "name_key"], keep="first")
recipes_final = recipes_final.loc[~same_page_and_name]
recipes_final["recipe_id"] = np.arange(1, len(recipes_final) + 1)                   # ids sin huecos
recipes_final = recipes_final[FINAL_COLUMNS].copy()
log_rows("Dataset final", len(recipes_final))

In [ ]:
r = recipes_final

assert r["recipe_id"].is_unique
assert r["name"].fillna("").str.strip().ne("").all(), "hay recetas sin nombre"
assert r["ingredients"].fillna("").str.strip().ne("").all(), "hay recetas sin ingredientes"
assert r["instructions"].fillna("").str.strip().ne("").all(), "hay recetas sin pasos"
assert r["total_time_min"].dropna().gt(0).all() and r["total_time_min"].dropna().le(12 * 60).all(), \
    "tiempo fuera de 0 a 720 min"
assert r["rating"].dropna().between(0, 5).all(), "valoración fuera de 0 a 5"
assert set(r["difficulty"].dropna()) <= {"facil", "intermedio", "reto"}
assert not r.duplicated(["url", "name"]).any()

# Etiquetas: solo las permitidas, y sin contradicciones.
used_tags = set(r["tags"].dropna().str.split(" | ", regex=False).explode())
assert used_tags <= set(ALLOWED_TAGS), used_tags - set(ALLOWED_TAGS)
tag_sets = r["tags"].fillna("").map(lambda s: set(s.split(" | ")))
assert not (tag_sets.map(lambda s: "Vegetariana" in s and bool(s & PROTEIN_TAGS))).any(), "vegetariana con carne"
assert not (tag_sets.map(lambda s: "Vegana" in s and "Vegetariana" not in s)).any(), "vegana sin ser vegetariana"

# Pasos: sin etiquetas de truco, sin "Paso 1", sin letras rotas.
steps_text = r["instructions"]
assert not steps_text.str.contains(r"(?:^|\| )(?:Truco|Consejo|Nota)\b", regex=True).any(), "quedaron trucos"
assert not steps_text.str.contains(r"\bPaso\s*\d", regex=True, case=False).any(), "quedó 'Paso N'"
assert not steps_text.str.contains("Ã").any(), "letras rotas en pasos"
assert not steps_text.str.contains(r"(?:^|\| )[¡!]?\s*Espero (?:que )?(?:os|te|les)\b",
                                   regex=True, case=False).any(), "quedaron frases de despedida"

# Texto limpio de verdad: nada de letras rotas, HTML, enlaces ni comillas de imprenta.
for col in ["name", "ingredients", "instructions"]:
    s = r[col].fillna("")
    assert not s.str.contains(r"[ÃÂ]|\ufffd", regex=True).any(), f"letras rotas en {col}"
    assert not s.str.contains(r"&[a-zA-Z#][a-zA-Z0-9]{1,8};", regex=True).any(), f"entidades HTML en {col}"
    assert not s.str.contains(r"https?://|www\.", regex=True).any(), f"enlaces en {col}"
    assert not s.str.contains(r"[“”«»‘’…–—´]", regex=True).any(), f"comillas/guiones de imprenta en {col}"
    assert not s.str.contains(r"[®™]", regex=True).any(), f"símbolos de marca en {col}"
    assert not s.str.contains(r"[½¼¾⅓⅔⅛]", regex=True).any(), f"fracciones sin convertir en {col}"
    assert not s.str.contains(r"[•▪●·→←⇒›‹¤¬¨˚\u00ad]", regex=True).any(), f"viñetas o flechas en {col}"

# Ingredientes: ni porciones disfrazadas de ingrediente, ni genéricos duplicados.
ing = r["ingredients"]
assert not ing.str.contains(r"(?:^|\| )\s*(?:Para|Rinde|Salen?)\s+\d+\s*"
                            r"(?:personas?|comensales|raciones?|porciones?)\s*(?:\||$)",
                            regex=True, case=False).any(), "quedaron notas de porciones"
for generico in ["caldo en polvo", "queso crema", "leche condensada", "polvo para hornear"]:
    assert not ing.str.contains(rf"\b{generico}\b(?:\s+\w+){{0,2}}\s+\b{generico}\b",
                                regex=True, case=False).any(), f"'{generico}' duplicado"

# Columnas que se eliminaron a propósito.
for removed in ["servings", "context", "meal_type", "category_tags", "app_category", "listed_time_min"]:
    assert removed not in r.columns, removed

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
recipes_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")


# Comprobación: el archivo se lee de vuelta igual.
check = pd.read_csv(OUTPUT_PATH)
assert check.shape == recipes_final.shape
assert check.columns.tolist() == FINAL_COLUMNS